# Consolidated Notebook -- Questions 1, 2 & 3

Merged from `consolidated_Q1.ipynb`, `consolidated_Q2.ipynb`, and
`consolidated_Q3.ipynb`. **Run top to bottom, in one kernel** -- Q2 and Q3
depend on the shared path constants (`DATA_DIR`, `RESULTS_DIR`, etc.)
defined in Q1's Config cell near the top, and on `GROUP2_ABT_CSV` /
`RESULTS_Q2_DIR` defined in the "Shared paths" cell right after Q1.

All file paths across all three sections have been repointed to assume this
notebook lives at the **project root** (same level as `data` and `src`),
matching the screenshot of your project structure. See the chat reply below
this for a full list of what changed and what's still worth double-checking
in each original source file.


# Question 1 -- AQMN Spatial-Coverage Optimization

Self-contained version of the pipeline in `src/analysis`. All logic from
`src/analysis/config.py`, `src/analysis/io_utils.py`, `src/analysis/grid.py`, `src/analysis/coverage.py`, `src/analysis/optimize.py`,
`src/analysis/allocation.py`, `src/analysis/metrics.py`, `src/analysis/visualize.py`, `src/analysis/check_inputs.py`, and
`src/analysis/run_models.py` is inlined below -- no dependency on the `src` package, so
this notebook runs on its own. The original modules in `src/analysis` are
unchanged and remain the source of truth for the CLI (`python -m
src.analysis.run_models`); this notebook is a convenience copy for
interactive / submission use.

**Run cells top to bottom.** Section order mirrors the module dependency
chain: config -> io_utils -> grid -> coverage -> optimize -> allocation ->
metrics -> visualize -> (optional) check_inputs -> run_models pipeline.

Outputs land in `data/results/n<N_NEW_STATIONS>/`, namespaced by station
count so re-running with a different `N_NEW_STATIONS` never overwrites a
previous run: per-model site tables (with lon/lat), per-model map PNGs, a
combined comparison PNG, and `model_comparison_n<N>.csv` (PCR + MARR).


## Imports

In [1]:
import time
from pathlib import Path
from typing import Optional, Iterable

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import pulp

import matplotlib
matplotlib.use("Agg")  # headless-safe backend; must be set before pyplot import
import matplotlib.pyplot as plt

try:
    from tqdm import tqdm
except ImportError:  # tqdm is optional -- falls back to a plain loop, no progress bar
    def tqdm(iterable, **kwargs):
        return iterable


## Methodology (overview)

This notebook builds an Analytical Base Table (ABT) of ~2 km candidate
monitoring sites across the Philippines, attaches each candidate's province,
built-up-land eligibility (from GHSL), and distance to the nearest existing
station. It then solves a Maximal Covering Location Problem (MCLP) four
times -- toggling *scope* (national vs. per-province budget) and
*eligibility* (built-up land required vs. not) -- to produce four candidate
siting plans. Each plan is scored on Population Coverage Ratio (PCR) gain
and a redundancy metric (MARR), then visualized on a national map.

## Config

Equivalent of `src/analysis/config.py`. `PROJECT_ROOT` assumes this notebook lives at the
project root (same level as `data` and `src`) -- adjust if you move it.


In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
GIS_DIR = DATA_DIR / "gis"

# Vector boundaries
ADMIN0_SHP = GIS_DIR / "phl_admin0.shp"
ADMIN2_SHP = GIS_DIR / "phl_admin2.shp"
ADMIN2_NAME_FIELD = "adm2_name"

# Population (1km density, ASCII XYZ -> loaded as CSV)
POP_DENSITY_CSV = GIS_DIR / "phl_pd_2020_1km_UNadj_ASCII_XYZ.csv"
POP_CSV_COLS = {"lon": "X", "lat": "Y", "density": "Z"}

# GHSL Built-up surface tiles (fraction built-up per cell, 3-arcsec ~ 100m, EPSG:4326)
GHSL_TILES = [
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R7_C31.tif",
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R8_C30.tif",
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R8_C31.tif",
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R9_C30.tif",
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R9_C31.tif",
]
GHSL_MOSAIC_TIF = GIS_DIR / "ghsl_built_mosaic_phl.tif"  # cached merged output

# Stations
STATION_LIST_CSV = DATA_DIR / "station_list.csv"
STATION_LAT_COL = "lat"
STATION_LON_COL = "lon"

# Province-level population/area benchmark (for validation + min-station rule)
PROVINCE_STATS_CSV = DATA_DIR / "Population_LandArea_Density_Province.csv"

# Cached outputs -- kept separate from any pre-existing analytical_base_table.parquet
ABT_PARQUET = DATA_DIR / "gis_base_table.parquet"
RESULTS_DIR = DATA_DIR / "results"

# --- CRS -----------------------------------------------------------------
CRS_WGS84 = "EPSG:4326"
CRS_PROJECTED = "EPSG:3123"  # PRS92 / Philippines Zone III (meters)

# --- Model parameters (editable) -----------------------------------------
RADIUS_KM = 4.0                # station monitoring radius
CANDIDATE_CELL_KM = 2.0        # candidate grid resolution
N_NEW_STATIONS = 279           # total new stations to place
NO_OVERLAP_KM = 8.0            # min distance from an existing station (2x radius)
BUILTUP_THRESHOLD = 0.2        # min built-up fraction for a candidate to be "eligible"
MIN_STATIONS_PER_PROVINCE = 1  # floor used when splitting the budget across provinces
SOLVER_TIME_LIMIT_SEC = 300


## io_utils -- loaders for boundaries, population, built-up raster, stations

In [3]:
def load_country_boundary() -> gpd.GeoDataFrame:
    gdf = gpd.read_file(ADMIN0_SHP)
    return gdf.to_crs(CRS_WGS84)


def load_provinces() -> gpd.GeoDataFrame:
    gdf = gpd.read_file(ADMIN2_SHP).to_crs(CRS_WGS84)
    if ADMIN2_NAME_FIELD not in gdf.columns:
        raise KeyError(
            f"'{ADMIN2_NAME_FIELD}' not found in phl_admin2 attributes.\n"
            f"Available columns: {list(gdf.columns)}\n"
            f"Update ADMIN2_NAME_FIELD in the Config cell above."
        )
    return gdf


def load_population_points() -> gpd.GeoDataFrame:
    """
    Loads the 1km population-density ASCII XYZ file and converts density -> counts.
    Returns a GeoDataFrame of points with columns:
        lon, lat, density, cell_area_km2, population, geometry
    """
    df = pd.read_csv(POP_DENSITY_CSV)
    lon_col = POP_CSV_COLS["lon"]
    lat_col = POP_CSV_COLS["lat"]
    dens_col = POP_CSV_COLS["density"]
    missing = [c for c in (lon_col, lat_col, dens_col) if c not in df.columns]
    if missing:
        raise KeyError(
            f"Column(s) {missing} not found in {POP_DENSITY_CSV.name}.\n"
            f"Available columns: {list(df.columns)}\n"
            f"Update POP_CSV_COLS in the Config cell above."
        )

    df = df.rename(columns={lon_col: "lon", lat_col: "lat", dens_col: "density"})
    df = df[df["density"] > 0].reset_index(drop=True)  # drop nodata / ocean cells

    # Per-row cell area in km^2 -- a "1km" lon/lat grid shrinks east-west as
    # |lat| grows, so area is NOT a flat 1 km^2 across the whole archipelago.
    lat_rad = np.radians(df["lat"].to_numpy())
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(lat_rad)
    cell_deg = 1.0 / km_per_deg_lat  # ~1km spacing expressed in degrees latitude
    df["cell_area_km2"] = (cell_deg * km_per_deg_lat) * (cell_deg * km_per_deg_lon)

    df["population"] = df["density"] * df["cell_area_km2"]

    gdf = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs=CRS_WGS84
    )
    return gdf


def build_ghsl_mosaic(force: bool = False) -> None:
    """Merges the GHSL built-up tiles once and caches the result to disk."""
    if GHSL_MOSAIC_TIF.exists() and not force:
        return
    srcs = [rasterio.open(p) for p in GHSL_TILES]
    mosaic, transform = merge(srcs)
    meta = srcs[0].meta.copy()
    meta.update(driver="GTiff", height=mosaic.shape[1], width=mosaic.shape[2], transform=transform)
    with rasterio.open(GHSL_MOSAIC_TIF, "w", **meta) as dst:
        dst.write(mosaic)
    for s in srcs:
        s.close()


def check_ghsl_covers_country(country: gpd.GeoDataFrame) -> bool:
    """Sanity check: does the merged GHSL mosaic bounding box contain the PH boundary?"""
    build_ghsl_mosaic()
    with rasterio.open(GHSL_MOSAIC_TIF) as src:
        left, bottom, right, top = src.bounds
    minx, miny, maxx, maxy = country.total_bounds
    covers = (left <= minx) and (bottom <= miny) and (right >= maxx) and (top >= maxy)
    if not covers:
        print(
            "[WARNING] GHSL mosaic does not fully cover the PH boundary.\n"
            f"  mosaic bounds : {(left, bottom, right, top)}\n"
            f"  country bounds: {(minx, miny, maxx, maxy)}\n"
            "  Candidate cells outside the mosaic will have no built-up value "
            "and will be marked ineligible by default -- check for a missing "
            "GHSL tile (e.g. Batanes, or Sulu/Tawi-Tawi)."
        )
    return covers


def load_stations() -> gpd.GeoDataFrame:
    df = pd.read_csv(STATION_LIST_CSV)
    missing = [c for c in (STATION_LON_COL, STATION_LAT_COL) if c not in df.columns]
    if missing:
        raise KeyError(
            f"Column(s) {missing} not found in {STATION_LIST_CSV.name}.\n"
            f"Available columns: {list(df.columns)}\n"
            f"Update STATION_LAT_COL / STATION_LON_COL in the Config cell above."
        )
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[STATION_LON_COL], df[STATION_LAT_COL]),
        crs=CRS_WGS84,
    )
    return gdf


### Method: GHSL Built-Up Surface (site eligibility screen)

- **Name:** GHS-BUILT-S (Global Human Settlement Layer, Built-up Surface)
- **Source / Documentation:** https://ghsl.jrc.ec.europa.eu/
- **Purpose:** Every candidate site is sampled against this raster to get a
  built-up-surface value. Sites above `BUILTUP_THRESHOLD` are flagged
  `is_eligible=True` and are the only sites the "Modified Framework" models
  (1 and 2) are allowed to choose from -- the idea being that monitoring
  stations belong on developed land, not open countryside or water.

## grid -- candidate monitoring-site grid, province + built-up attributes

In [4]:
def make_candidate_grid(country: gpd.GeoDataFrame, cell_km: float = CANDIDATE_CELL_KM) -> gpd.GeoDataFrame:
    """Generates a regular grid of candidate points at `cell_km` spacing, clipped to the country boundary."""
    country_proj = country.to_crs(CRS_PROJECTED)
    minx, miny, maxx, maxy = country_proj.total_bounds
    step_m = cell_km * 1000

    xs = np.arange(minx, maxx, step_m)
    ys = np.arange(miny, maxy, step_m)
    xx, yy = np.meshgrid(xs, ys)
    pts = gpd.GeoSeries([Point(x, y) for x, y in zip(xx.ravel(), yy.ravel())], crs=CRS_PROJECTED)

    grid = gpd.GeoDataFrame(geometry=pts)
    union = country_proj.geometry.union_all()
    grid = grid[grid.within(union)].reset_index(drop=True)
    grid["candidate_id"] = grid.index

    return grid.to_crs(CRS_WGS84)


def attach_province(points: gpd.GeoDataFrame, provinces: gpd.GeoDataFrame,
                     province_col: str = "province") -> gpd.GeoDataFrame:
    """
    Generic spatial join: tags any point GeoDataFrame (candidate grid OR
    population points) with the province it falls in. Points that don't land
    exactly inside a polygon fall back to nearest-province matching.
    """
    original_index = points.index

    joined = gpd.sjoin(points, provinces[[ADMIN2_NAME_FIELD, "geometry"]],
                        how="left", predicate="within")
    joined = joined.rename(columns={ADMIN2_NAME_FIELD: province_col}).drop(columns=["index_right"])
    joined = joined[~joined.index.duplicated(keep="first")]
    joined = joined.reindex(original_index)

    missing = joined[province_col].isna()
    if missing.any():
        points_proj = points.loc[missing, ["geometry"]].to_crs(CRS_PROJECTED)
        provinces_proj = provinces[[ADMIN2_NAME_FIELD, "geometry"]].to_crs(CRS_PROJECTED)

        nearest = gpd.sjoin_nearest(points_proj, provinces_proj, how="left")
        nearest = nearest[~nearest.index.duplicated(keep="first")]
        nearest = nearest.reindex(points_proj.index)

        joined.loc[missing, province_col] = nearest[ADMIN2_NAME_FIELD]

    return joined


def attach_builtup(grid: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Samples the GHSL built-up fraction at each candidate point location."""
    with rasterio.open(GHSL_MOSAIC_TIF) as src:
        coords = [(p.x, p.y) for p in grid.geometry]
        values = np.array([v[0] for v in src.sample(coords)], dtype=float)

    grid = grid.copy()
    grid["builtup_frac"] = values
    grid["is_eligible"] = grid["builtup_frac"] >= BUILTUP_THRESHOLD
    return grid


### Method: Ball Tree Nearest-Neighbor Search (haversine metric)

- **Name:** Ball Tree spatial index, haversine distance
- **Documentation:** https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.BallTree.html
- **Purpose:** Used everywhere this pipeline needs "which points are within
  X km of which other points" at national scale (existing-station coverage,
  candidate-to-existing-station distance, candidate-to-population
  reachability). The haversine metric computes true great-circle distance on
  a sphere directly from lat/lon, avoiding the distortion a flat-Cartesian
  distance would introduce across the archipelago's latitude range.

## coverage -- spatial coverage relationships (BallTree / haversine)

In [5]:
EARTH_RADIUS_KM = 6371.0088  # mean Earth radius (IUGG)


def _to_radians(gdf: gpd.GeoDataFrame) -> np.ndarray:
    lat = gdf.geometry.y.to_numpy()
    lon = gdf.geometry.x.to_numpy()
    return np.radians(np.column_stack([lat, lon]))


def mark_existing_coverage(pop_points: gpd.GeoDataFrame, stations: gpd.GeoDataFrame,
                            radius_km: float = RADIUS_KM) -> gpd.GeoDataFrame:
    """Flags each population point as covered (True) or not by the CURRENT network."""
    tree = BallTree(_to_radians(stations), metric="haversine")
    idx = tree.query_radius(_to_radians(pop_points), r=radius_km / EARTH_RADIUS_KM)
    pop_points = pop_points.copy()
    pop_points["covered_existing"] = [len(i) > 0 for i in idx]
    return pop_points


def drop_candidates_near_existing(grid: gpd.GeoDataFrame, stations: gpd.GeoDataFrame,
                                   min_dist_km: float = NO_OVERLAP_KM) -> gpd.GeoDataFrame:
    """Removes candidate sites within `min_dist_km` of any existing station."""
    tree = BallTree(_to_radians(stations), metric="haversine")
    dist, _ = tree.query(_to_radians(grid), k=1)
    dist_km = dist.ravel() * EARTH_RADIUS_KM
    grid = grid.copy()
    grid["dist_to_existing_km"] = dist_km
    return grid[dist_km >= min_dist_km].reset_index(drop=True)


def build_coverage_sets(grid: gpd.GeoDataFrame, pop_points: gpd.GeoDataFrame,
                         radius_km: float = RADIUS_KM):
    """
    Returns:
        uncovered : subset of pop_points NOT already covered by the existing
                    network (reset to a fresh 0..n-1 index -- this index is
                    what N_j's keys refer to)
        N_j       : dict {row position in `uncovered` -> [candidate_ids within radius_km]}
    """
    uncovered = pop_points[~pop_points["covered_existing"]].reset_index(drop=True)
    tree = BallTree(_to_radians(grid), metric="haversine")
    idx_lists = tree.query_radius(_to_radians(uncovered), r=radius_km / EARTH_RADIUS_KM)

    candidate_ids = grid["candidate_id"].to_numpy()
    N_j = {j: candidate_ids[idxs].tolist() for j, idxs in enumerate(idx_lists) if len(idxs) > 0}
    return uncovered, N_j


### Model: Maximal Covering Location Problem (MCLP)

- **Name:** Maximal Covering Location Problem, solved as a Mixed-Integer
  Linear Program (MILP)
- **Solver documentation:** https://coin-or.github.io/pulp/ (PuLP, using the
  free CBC solver)
- **Purpose:** Given a fixed number of new sites to place, choose the subset
  that maximizes population newly brought within `RADIUS_KM` of a station.
  This is the core optimization used by all four models; they differ only in
  *which* candidate sites are eligible (`use_eligibility`) and *what scope*
  each solve is run over (national in one shot vs. one solve per province).

## optimize -- maximal-covering-location MILP (mirrors paper eq. 4-6)

In [6]:
def solve_mclp(grid: pd.DataFrame, uncovered_pop: pd.DataFrame, N_j: dict,
                n_new: int, use_eligibility: bool = True,
                candidate_subset: Optional[Iterable[int]] = None,
                time_limit_sec: int = SOLVER_TIME_LIMIT_SEC):
    """
    Solves:
        maximize   sum_j w_j * z_j
        subject to sum_i x_i == n_new
                   z_j <= sum_{i in N_j} x_i      for every population point j
                   x_i == 0 for ineligible i      (only if use_eligibility)
                   x_i, z_j in {0, 1}

    Returns: chosen_ids, newly_covered_pop, status
    """
    cand_df = grid if candidate_subset is None else grid[grid["candidate_id"].isin(candidate_subset)]
    candidate_ids = cand_df["candidate_id"].tolist()

    if n_new > len(candidate_ids):
        print(
            f"[WARNING] solve_mclp: requested n_new={n_new} but only "
            f"{len(candidate_ids)} candidate sites are available in this subset "
            f"-- capping to {len(candidate_ids)}. If this is unexpected, check "
            f"that province budgets were computed with the correct "
            f"max_per_province capacity for this eligibility setting."
        )
        n_new = len(candidate_ids)

    prob = pulp.LpProblem("mclp", pulp.LpMaximize)
    x = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in candidate_ids}

    relevant_j = {j: [i for i in ids if i in x] for j, ids in N_j.items()}
    relevant_j = {j: ids for j, ids in relevant_j.items() if ids}
    z = {j: pulp.LpVariable(f"z_{j}", cat="Binary") for j in relevant_j}

    w = uncovered_pop["population"].to_dict()
    prob += pulp.lpSum(w[j] * z[j] for j in z)               # objective (eq. 4)
    prob += pulp.lpSum(x.values()) == n_new                  # station budget (eq. 5)

    for j, ids in relevant_j.items():
        prob += z[j] <= pulp.lpSum(x[i] for i in ids)        # coverage link

    if use_eligibility:
        elig = cand_df.set_index("candidate_id")["is_eligible"]
        for i in candidate_ids:
            if not bool(elig.get(i, False)):
                prob += x[i] == 0                            # impervious constraint

    solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit_sec)
    prob.solve(solver)

    chosen_ids = [i for i, var in x.items() if pulp.value(var) is not None and pulp.value(var) > 0.5]
    newly_covered = sum(w[j] for j in z if pulp.value(z[j]) is not None and pulp.value(z[j]) > 0.5)
    return chosen_ids, newly_covered, pulp.LpStatus[prob.status]


## allocation -- province budgets (capacity-aware, with feasibility guards)

In [7]:
def compute_province_capacity(grid: pd.DataFrame, use_eligibility: bool,
                               province_col: str = "province") -> dict:
    """
    Counts available candidate sites per province, matching the same scoping
    a province-level solve_mclp() call would see: eligible-only sites when
    use_eligibility=True (Model 2), all sites when False (Model 3).
    """
    df = grid[grid["is_eligible"]] if use_eligibility else grid
    return df.groupby(province_col)["candidate_id"].nunique().to_dict()


def compute_province_budgets(uncovered_pop_with_province: pd.DataFrame,
                              total_new_stations: int = N_NEW_STATIONS,
                              min_per_province: int = MIN_STATIONS_PER_PROVINCE,
                              max_per_province: Optional[dict] = None) -> dict:
    """
    Splits total_new_stations across provinces proportional to each
    province's share of currently-uncovered population.

    max_per_province : optional dict {province_name: max_n_new_stations}, e.g.
        the count of available (eligible) candidate sites in that province.
        Provinces absent from this dict are treated as uncapped.

    Raises ValueError up front if total_new_stations can't be reconciled with
    min_per_province or max_per_province, instead of looping forever.
    """
    prov_pop = uncovered_pop_with_province.groupby("province")["population"].sum()
    provinces = prov_pop.index.tolist()
    n_provinces = len(provinces)

    floor_total = min_per_province * n_provinces
    if total_new_stations < floor_total:
        raise ValueError(
            f"total_new_stations ({total_new_stations}) is less than "
            f"min_per_province ({min_per_province}) x n_provinces ({n_provinces}) "
            f"= {floor_total}. Raise N_NEW_STATIONS, lower MIN_STATIONS_PER_PROVINCE, "
            f"or reduce the number of provinces in scope."
        )

    if max_per_province is not None:
        total_capacity = sum(max_per_province.get(p, float("inf")) for p in provinces)
        if total_new_stations > total_capacity:
            raise ValueError(
                f"total_new_stations ({total_new_stations}) exceeds total candidate "
                f"capacity across provinces ({total_capacity:,.0f}). Lower N_NEW_STATIONS "
                f"or widen the candidate grid (e.g. smaller CANDIDATE_CELL_KM)."
            )
        capacity_floor = sum(min(max_per_province.get(p, float("inf")), min_per_province)
                              for p in provinces)
        if capacity_floor < floor_total:
            short = [p for p in provinces
                     if max_per_province.get(p, float("inf")) < min_per_province]
            raise ValueError(
                f"min_per_province ({min_per_province}) exceeds available candidate "
                f"capacity in: {short}. Lower MIN_STATIONS_PER_PROVINCE or expand the "
                f"candidate grid for those provinces."
            )

    shares = prov_pop / prov_pop.sum()

    budgets = (shares * total_new_stations).round().astype(int)
    budgets = budgets.clip(lower=min_per_province)

    if max_per_province is not None:
        for p in provinces:
            cap = max_per_province.get(p)
            if cap is not None:
                budgets[p] = min(budgets[p], cap)

    # Rounding (and capping) can drift the total off target; nudge the
    # largest-share provinces up/down by 1 until it matches exactly.
    diff = total_new_stations - budgets.sum()
    if diff != 0:
        order = shares.sort_values(ascending=False).index.tolist()
        i = 0
        stall_guard = 0
        max_stalls = len(order) + 1
        while diff != 0:
            prov = order[i % len(order)]
            step = 1 if diff > 0 else -1

            blocked = False
            if step < 0 and budgets[prov] <= min_per_province:
                blocked = True
            if step > 0 and max_per_province is not None:
                cap = max_per_province.get(prov)
                if cap is not None and budgets[prov] >= cap:
                    blocked = True

            if blocked:
                i += 1
                stall_guard += 1
                if stall_guard > max_stalls:
                    raise RuntimeError(
                        "compute_province_budgets: could not converge on a feasible "
                        "allocation despite passing feasibility checks -- this is a bug, "
                        "please report the inputs that triggered it."
                    )
                continue

            budgets[prov] += step
            diff -= step
            i += 1
            stall_guard = 0

    return budgets.to_dict()


## metrics -- PCR before/after, MARR, model comparison table

In [8]:
def pcr_before(pop_points: pd.DataFrame) -> float:
    total = pop_points["population"].sum()
    covered = pop_points.loc[pop_points["covered_existing"], "population"].sum()
    return covered / total


def pcr_after(pop_points: pd.DataFrame, newly_covered_population: float) -> float:
    total = pop_points["population"].sum()
    covered_before = pop_points.loc[pop_points["covered_existing"], "population"].sum()
    return (covered_before + newly_covered_population) / total


def compute_marr(pop_points: gpd.GeoDataFrame, all_stations: gpd.GeoDataFrame,
                  radius_km: float = RADIUS_KM) -> float:
    """
    Monitoring Area Repetition Rate (MARR): population-weighted average number
    of REDUNDANT stations covering an already-covered population point, using
    the full network (existing + newly placed stations for this model).

        MARR = sum_p[ pop_p * (n_stations_covering_p - 1) ] / sum_p[ pop_p ]
               over points p covered by >= 1 station

    0.0 -> every covered point is served by exactly one station (no overlap)
    1.0 -> covered points are served by two stations on average
    etc.

    WORKING DEFINITION -- MARR is not otherwise specified in this codebase.
    This mirrors pcr_after's convention of scoring the combined existing+new
    network, population-weighted like the rest of this section. Confirm
    against your source formula and adjust this function if the definition
    differs -- everything that calls it only depends on the float it returns.
    """
    if len(all_stations) == 0 or len(pop_points) == 0:
        return 0.0

    tree = BallTree(_to_radians(all_stations), metric="haversine")
    idx_lists = tree.query_radius(
        _to_radians(pop_points), r=radius_km / EARTH_RADIUS_KM
    )
    counts = np.array([len(i) for i in idx_lists])

    covered_mask = counts >= 1
    pop = pop_points["population"].to_numpy()
    covered_pop = pop[covered_mask]
    covered_counts = counts[covered_mask]

    total_covered_pop = covered_pop.sum()
    if total_covered_pop == 0:
        return 0.0

    return float(np.sum(covered_pop * (covered_counts - 1)) / total_covered_pop)


def summarize_models(results: dict, pop_points: pd.DataFrame,
                      marr_values: Optional[dict] = None) -> pd.DataFrame:
    """
    results     : {model_name: {"chosen_ids": [...], "newly_covered_pop": float}}
    marr_values : optional {model_name: float}, from compute_marr() per model.
    """
    base_pcr = pcr_before(pop_points)
    rows = []
    for name, r in results.items():
        row = {
            "model": name,
            "n_stations_placed": len(r["chosen_ids"]),
            "newly_covered_population": r["newly_covered_pop"],
            "pcr_before": base_pcr,
            "pcr_after": pcr_after(pop_points, r["newly_covered_pop"]),
            "pcr_gain_pp": (pcr_after(pop_points, r["newly_covered_pop"]) - base_pcr) * 100,
        }
        if marr_values is not None:
            row["marr"] = marr_values.get(name)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("pcr_after", ascending=False).reset_index(drop=True)


## visualize -- static maps of chosen sites vs existing network (matplotlib + geopandas)

In [9]:
def _site_points(chosen_df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Builds a point GeoDataFrame from a chosen-sites table's lon/lat columns."""
    return gpd.GeoDataFrame(
        chosen_df,
        geometry=gpd.points_from_xy(chosen_df["lon"], chosen_df["lat"]),
        crs=CRS_WGS84,
    )


def plot_model_sites(model_name: str, chosen_df: pd.DataFrame,
                      country: gpd.GeoDataFrame, stations: gpd.GeoDataFrame,
                      out_path) -> None:
    """Saves one PNG: country outline + existing stations + this model's new sites."""
    sites = _site_points(chosen_df)

    fig, ax = plt.subplots(figsize=(8, 10))
    country.boundary.plot(ax=ax, color="black", linewidth=0.5)
    stations.plot(ax=ax, color="gray", markersize=6, label=f"Existing ({len(stations)})")
    sites.plot(ax=ax, color="crimson", markersize=8, label=f"New ({len(sites)})")
    ax.set_title(model_name, fontsize=11)
    ax.set_axis_off()
    ax.legend(loc="lower left", fontsize=8, frameon=True)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_all_models(results: dict, grid_df: pd.DataFrame, country: gpd.GeoDataFrame,
                     stations: gpd.GeoDataFrame, out_path) -> None:
    """Saves one PNG with a 2x2 panel comparing every model's chosen sites side by side."""
    n = len(results)
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 8 * nrows))
    axes = axes.ravel() if n > 1 else [axes]

    for ax, (name, r) in zip(axes, results.items()):
        chosen_df = grid_df[grid_df["candidate_id"].isin(r["chosen_ids"])]
        sites = _site_points(chosen_df)
        country.boundary.plot(ax=ax, color="black", linewidth=0.4)
        stations.plot(ax=ax, color="gray", markersize=3)
        sites.plot(ax=ax, color="crimson", markersize=4)
        ax.set_title(f"{name}\n({len(sites)} new stations)", fontsize=9)
        ax.set_axis_off()

    for ax in axes[n:]:  # hide any unused panels if results count is odd
        ax.set_visible(False)

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


## check_inputs (optional)

Run this section before the main pipeline the first time, or after editing
any config constant above, to confirm file paths and column names resolve
correctly. Safe to skip on later runs.

In [10]:
def check_admin2():
    print("=" * 70)
    print(f"phl_admin2.shp  ->  {ADMIN2_SHP}")
    gdf = gpd.read_file(ADMIN2_SHP)
    print(f"  rows: {len(gdf)}")
    print(f"  columns: {list(gdf.columns)}")
    print(f"  CRS: {gdf.crs}")
    print("  first 3 rows (non-geometry columns):")
    print(gdf.drop(columns="geometry").head(3).to_string())
    print(f"\n  ==> current ADMIN2_NAME_FIELD = '{ADMIN2_NAME_FIELD}' "
          f"{'FOUND' if ADMIN2_NAME_FIELD in gdf.columns else '*** NOT FOUND ***'}")


def check_population_csv():
    print("=" * 70)
    print(f"population CSV  ->  {POP_DENSITY_CSV}")
    df = pd.read_csv(POP_DENSITY_CSV, nrows=5)
    print(f"  columns: {list(df.columns)}")
    print("  first 5 rows:")
    print(df.to_string())
    for key, col in POP_CSV_COLS.items():
        status = "FOUND" if col in df.columns else "*** NOT FOUND ***"
        print(f"  ==> current POP_CSV_COLS['{key}'] = '{col}'  {status}")


def check_stations():
    print("=" * 70)
    print(f"station list  ->  {STATION_LIST_CSV}")
    df = pd.read_csv(STATION_LIST_CSV, nrows=5)
    print(f"  columns: {list(df.columns)}")
    print("  first 5 rows:")
    print(df.to_string())
    for key, col in [("STATION_LAT_COL", STATION_LAT_COL), ("STATION_LON_COL", STATION_LON_COL)]:
        status = "FOUND" if col in df.columns else "*** NOT FOUND ***"
        print(f"  ==> current {key} = '{col}'  {status}")


def check_admin0():
    print("=" * 70)
    print(f"phl_admin0.shp  ->  {ADMIN0_SHP}")
    gdf = gpd.read_file(ADMIN0_SHP)
    print(f"  rows: {len(gdf)}, CRS: {gdf.crs}")
    print(f"  bounds: {gdf.total_bounds}")


def check_ghsl_tiles():
    print("=" * 70)
    print("GHSL built-up tiles:")
    for p in GHSL_TILES:
        exists = p.exists()
        print(f"  {p.name}  exists={exists}")
        if exists:
            with rasterio.open(p) as src:
                print(f"    bounds={src.bounds}, crs={src.crs}, shape={src.shape}")


def run_input_checks():
    check_admin0()
    check_admin2()
    check_population_csv()
    check_stations()
    check_ghsl_tiles()
    print("=" * 70)
    print("Done. Fix any '*** NOT FOUND ***' lines above by editing the "
          "matching constant in the Config cell, then re-run this cell to "
          "confirm before moving on to the main pipeline below.")


In [11]:
# Uncomment to run input checks (recommended on first run, or after editing config):
# run_input_checks()


## run_models -- main pipeline

Set `REBUILD_ABT = True` below to force rebuilding the candidate grid/ABT
even if a cached parquet exists (needed after changing `CANDIDATE_CELL_KM` or
`NO_OVERLAP_KM` in the Config cell). `BUILTUP_THRESHOLD` is always safe to
change without a rebuild -- eligibility is recomputed from cached
`builtup_frac` every time.

In [12]:
REBUILD_ABT = False  # set True to force a fresh grid/ABT build


def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    return f"{seconds / 60:.1f}min"


class _Stage:
    """Small context manager that prints how long a pipeline stage took."""
    def __init__(self, label: str):
        self.label = label

    def __enter__(self):
        print(f"{self.label}...")
        self._t0 = time.time()
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            print(f"  done in {_fmt_elapsed(time.time() - self._t0)}")


def build_abt(force_rebuild: bool = False):
    """
    Builds (or loads a cached) analytical base table: the candidate grid with
    province, built-up eligibility, and distance-to-existing-station attached.
    Cached as GeoParquet at ABT_PARQUET after the first build.
    """
    provinces = load_provinces()
    stations = load_stations()

    if ABT_PARQUET.exists() and not force_rebuild:
        print(f"Loading cached ABT from {ABT_PARQUET} (set REBUILD_ABT = True to force a rebuild)")
        grid = gpd.read_parquet(ABT_PARQUET)
        grid["is_eligible"] = grid["builtup_frac"] >= BUILTUP_THRESHOLD
        return grid, stations, provinces

    print("No cached ABT found (or rebuild forced) -- building candidate grid from scratch...")
    country = load_country_boundary()
    check_ghsl_covers_country(country)

    grid = make_candidate_grid(country)
    grid = attach_province(grid, provinces)
    grid = attach_builtup(grid)
    grid = drop_candidates_near_existing(grid, stations)

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    grid.to_parquet(ABT_PARQUET)  # GeoParquet -- keeps geometry + CRS
    return grid, stations, provinces


def prepare_population(stations: gpd.GeoDataFrame, provinces: gpd.GeoDataFrame):
    pop_points = load_population_points()
    pop_points = mark_existing_coverage(pop_points, stations)
    pop_points = attach_province(pop_points, provinces)
    return pop_points


def run_model_1_national_modified(grid, uncovered_pop, N_j):
    chosen, new_pop, status = solve_mclp(
        grid, uncovered_pop, N_j, n_new=N_NEW_STATIONS, use_eligibility=True
    )
    print(f"[Model 1] status={status}, stations={len(chosen)}, newly_covered_pop={new_pop:,.0f}")
    return {"chosen_ids": chosen, "newly_covered_pop": new_pop}


def run_model_4_national_simple(grid, uncovered_pop, N_j):
    chosen, new_pop, status = solve_mclp(
        grid, uncovered_pop, N_j, n_new=N_NEW_STATIONS, use_eligibility=False
    )
    print(f"[Model 4] status={status}, stations={len(chosen)}, newly_covered_pop={new_pop:,.0f}")
    return {"chosen_ids": chosen, "newly_covered_pop": new_pop}


def run_model_by_province(grid, uncovered_pop, N_j, province_budgets: dict,
                           use_eligibility: bool, label: str):
    all_chosen, total_new_pop = [], 0.0
    for province, n_new in tqdm(province_budgets.items(), total=len(province_budgets),
                                 desc=f"[{label}] provinces", unit="province"):
        subset_ids = grid.loc[grid["province"] == province, "candidate_id"]
        if n_new <= 0 or subset_ids.empty:
            continue
        chosen, new_pop, status = solve_mclp(
            grid, uncovered_pop, N_j, n_new=n_new,
            use_eligibility=use_eligibility, candidate_subset=subset_ids,
        )
        if status != "Optimal":
            print(f"  [{label}] {province}: status={status} (n_new={n_new})")
        all_chosen.extend(chosen)
        total_new_pop += new_pop
    print(f"[{label}] stations={len(all_chosen)}, newly_covered_pop={total_new_pop:,.0f}")
    return {"chosen_ids": all_chosen, "newly_covered_pop": total_new_pop}


### Step 1 -- Build ABT (candidate grid, province tags, built-up eligibility)

In [13]:
run_start = time.time()

with _Stage("Building ABT (candidate grid, province tags, built-up eligibility)"):
    grid, stations, provinces = build_abt(force_rebuild=REBUILD_ABT)
    # Keep lon/lat as plain columns before dropping geometry -- otherwise every
    # downstream CSV (including the chosen-site tables) loses the coordinates
    # entirely, since geometry is what actually carries them.
    grid_df = pd.DataFrame(
        grid.assign(lon=grid.geometry.x, lat=grid.geometry.y).drop(columns="geometry")
    )

grid_df.head()


Building ABT (candidate grid, province tags, built-up eligibility)...
Loading cached ABT from /Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/data/gis_base_table.parquet (set REBUILD_ABT = True to force a rebuild)
  done in 0.7s


,candidate_id,province,builtup_frac,is_eligible,dist_to_existing_km,lon,lat
0,0,Tawi-Tawi,0.0,False,743.038686,119.395532,4.641376
1,1,Tawi-Tawi,1242.0,True,742.282756,119.395491,4.659456
2,2,Tawi-Tawi,0.0,False,734.882999,119.467568,4.659619
3,3,Tawi-Tawi,0.0,False,741.531502,119.395449,4.677536
4,4,Tawi-Tawi,0.0,False,734.124183,119.467529,4.677699


### Step 2 -- Load population points, mark existing coverage

In [14]:
with _Stage("Loading population points and marking existing coverage"):
    pop_points = prepare_population(stations, provinces)

pop_points.head()


Loading population points and marking existing coverage...
  done in 2.5min


,lon,lat,density,cell_area_km2,population,geometry,covered_existing,province
0,121.927916,21.070417,11.363080,0.933139,10.603336,POINT (121.92792 21.07042),False,Batanes
1,121.936250,21.070417,9.669359,0.933139,9.022859,POINT (121.93625 21.07042),False,Batanes
2,121.927916,21.062083,10.205834,0.933192,9.523999,POINT (121.92792 21.06208),False,Batanes
3,121.936250,21.062083,9.299525,0.933192,8.678239,POINT (121.93625 21.06208),False,Batanes
4,121.911250,20.937083,8.517798,0.933973,7.955397,POINT (121.91125 20.93708),False,Batanes


### Step 3 -- Build candidate <-> population coverage sets

In [15]:
with _Stage("Building candidate<->population coverage sets"):
    uncovered_pop, N_j = build_coverage_sets(grid, pop_points)
    uncovered_pop_df = pd.DataFrame(uncovered_pop.drop(columns="geometry"))

uncovered_pop_df.head()


Building candidate<->population coverage sets...
  done in 1.4s


,lon,lat,density,cell_area_km2,population,covered_existing,province
0,121.927916,21.070417,11.363080,0.933139,10.603336,False,Batanes
1,121.936250,21.070417,9.669359,0.933139,9.022859,False,Batanes
2,121.927916,21.062083,10.205834,0.933192,9.523999,False,Batanes
3,121.936250,21.062083,9.299525,0.933192,8.678239,False,Batanes
4,121.911250,20.937083,8.517798,0.933973,7.955397,False,Batanes


### Method: Largest Remainder Apportionment (province budget splitting)

- **Name:** Largest Remainder Method (a.k.a. Hamilton's Method), adapted for
  station-budget apportionment with a floor and capacity cap
- **Purpose:** Splits the national station budget across provinces in
  proportion to each province's share of currently-uncovered population,
  then nudges the largest-share provinces up or down by one until the total
  matches exactly -- while respecting a per-province minimum
  (`MIN_STATIONS_PER_PROVINCE`) and, for the province-level models, a
  capacity cap so no province is assigned more stations than it has eligible
  candidate sites for. This is what makes Models 2 and 3 "equitable" rather
  than a single unconstrained national pot.

### Step 4 -- Province budgets

Model 2 (eligibility-filtered) and Model 3 (unfiltered) see different candidate pools per province, so each gets its own capacity-aware budget.

In [16]:
with _Stage("Computing province budgets"):
    capacity_modified = compute_province_capacity(grid_df, use_eligibility=True)
    capacity_simple = compute_province_capacity(grid_df, use_eligibility=False)

    province_budgets_modified = compute_province_budgets(
        uncovered_pop_df, max_per_province=capacity_modified
    )
    province_budgets_simple = compute_province_budgets(
        uncovered_pop_df, max_per_province=capacity_simple
    )


Computing province budgets...
  done in 0.0s


### Step 5 -- Run all four models

In [17]:
results = {}
with _Stage("[Model 1] solving (national, modified framework)"):
    results["Model 1: National + Modified Framework"] = run_model_1_national_modified(
        grid_df, uncovered_pop_df, N_j
    )
with _Stage("[Model 2] solving (per-province, modified framework)"):
    results["Model 2: Province + Modified Framework"] = run_model_by_province(
        grid_df, uncovered_pop_df, N_j, province_budgets_modified,
        use_eligibility=True, label="Model 2",
    )
with _Stage("[Model 3] solving (per-province, simple distance)"):
    results["Model 3: Province + Simple Distance"] = run_model_by_province(
        grid_df, uncovered_pop_df, N_j, province_budgets_simple,
        use_eligibility=False, label="Model 3",
    )
with _Stage("[Model 4] solving (national, simple distance)"):
    results["Model 4: National + Simple Distance"] = run_model_4_national_simple(
        grid_df, uncovered_pop_df, N_j
    )


[Model 1] solving (national, modified framework)...
[Model 1] status=Optimal, stations=279, newly_covered_pop=42,595,244
  done in 69.2min
[Model 2] solving (per-province, modified framework)...


[Model 2] provinces:  57%|█████▋    | 50/87 [00:28<00:16,  2.20province/s]

[WARNING] solve_mclp: requested n_new=2 but only 1 candidate sites are available in this subset -- capping to 1. If this is unexpected, check that province budgets were computed with the correct max_per_province capacity for this eligibility setting.


[Model 2] provinces:  60%|█████▉    | 52/87 [00:28<00:10,  3.39province/s]

  [Model 2] Metropolitan Manila Second District: status=Infeasible (n_new=2)


[Model 2] provinces: 100%|██████████| 87/87 [00:46<00:00,  1.87province/s]


[Model 2] stations=275, newly_covered_pop=39,862,194
  done in 46.6s
[Model 3] solving (per-province, simple distance)...


[Model 3] provinces: 100%|██████████| 87/87 [00:57<00:00,  1.51province/s]


[Model 3] stations=277, newly_covered_pop=41,885,804
  done in 57.7s
[Model 4] solving (national, simple distance)...
[Model 4] status=Optimal, stations=279, newly_covered_pop=43,710,508
  done in 6.5min


### Metrics: Population Coverage Ratio (PCR) and Monitoring Area Repetition Rate (MARR)

- **PCR (Population Coverage Ratio):** share of national population within
  `RADIUS_KM` of *some* station. Reported before (existing network only) and
  after (existing + newly placed) each model.
- **MARR (Monitoring Area Repetition Rate) -- working definition:** not a
  standard named metric from external literature; defined for this pipeline
  as the population-weighted average number of *redundant* stations covering
  an already-covered point (0 = no overlap, 1 = covered twice on average).
  See `compute_marr()`'s docstring for the full formula. **Verify this
  matches your source paper's definition of MARR before citing it as
  established methodology** -- if your paper defines it differently, only
  this function needs to change.

### Step 6 -- Compute MARR per model

Uses the full network (existing + this model's new sites) for each model.

In [18]:
with _Stage("Computing MARR (Monitoring Area Repetition Rate) per model"):
    marr_values = {}
    for name, r in results.items():
        chosen_df = grid_df[grid_df["candidate_id"].isin(r["chosen_ids"])]
        new_sites = gpd.GeoDataFrame(
            chosen_df,
            geometry=gpd.points_from_xy(chosen_df["lon"], chosen_df["lat"]),
            crs=CRS_WGS84,
        )
        all_stations_for_model = gpd.GeoDataFrame(
            pd.concat([stations[["geometry"]], new_sites[["geometry"]]], ignore_index=True),
            geometry="geometry", crs=CRS_WGS84,
        )
        marr_values[name] = compute_marr(pop_points, all_stations_for_model)

marr_values


Computing MARR (Monitoring Area Repetition Rate) per model...
  done in 1.5s


{'Model 1: National + Modified Framework': 1.3285257371776562,
 'Model 2: Province + Modified Framework': 1.3494228575593128,
 'Model 3: Province + Simple Distance': 1.3205141371505025,
 'Model 4: National + Simple Distance': 1.2761419809972914}

### Step 7 -- Summarize, write results and visualizations

Outputs land in `data/results/n<N_NEW_STATIONS>/`, namespaced by station count.

In [19]:
summary = summarize_models(results, pop_points, marr_values=marr_values)

run_tag = f"n{N_NEW_STATIONS}"
run_results_dir = RESULTS_DIR / run_tag
run_results_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(run_results_dir / f"model_comparison_{run_tag}.csv", index=False)

with _Stage("Writing chosen-site tables and map visualizations"):
    country = load_country_boundary()
    for name, r in results.items():
        chosen_df = grid_df[grid_df["candidate_id"].isin(r["chosen_ids"])]
        safe_name = name.split(":")[0].replace(" ", "_").lower()
        chosen_df.to_csv(run_results_dir / f"{safe_name}_{run_tag}_sites.csv", index=False)
        plot_model_sites(
            name, chosen_df, country, stations,
            out_path=run_results_dir / f"{safe_name}_{run_tag}_map.png",
        )
    plot_all_models(
        results, grid_df, country, stations,
        out_path=run_results_dir / f"all_models_{run_tag}_comparison.png",
    )

print("\n=== Model comparison ===")
print(summary.to_string(index=False))
print(f"\nWritten to {run_results_dir}")
print(f"Total runtime: {_fmt_elapsed(time.time() - run_start)}")

summary


Writing chosen-site tables and map visualizations...
  done in 1.5min

=== Model comparison ===
                                 model  n_stations_placed  newly_covered_population  pcr_before  pcr_after  pcr_gain_pp     marr
   Model 4: National + Simple Distance                279              4.371051e+07     0.12679   0.418107    29.131777 1.276142
Model 1: National + Modified Framework                279              4.259524e+07     0.12679   0.410674    28.388486 1.328526
   Model 3: Province + Simple Distance                277              4.188580e+07     0.12679   0.405946    27.915665 1.320514
Model 2: Province + Modified Framework                275              3.986219e+07     0.12679   0.392459    26.566988 1.349423

Written to /Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/data/results/n279
Total runtime: 81.5min


,model,n_stations_placed,newly_covered_population,pcr_before,pcr_after,pcr_gain_pp,marr
0,Model 4: National + Simple Distance,279,4.371051e+07,0.12679,0.418107,29.131777,1.276142
1,Model 1: National + Modified Framework,279,4.259524e+07,0.12679,0.410674,28.388486,1.328526
2,Model 3: Province + Simple Distance,277,4.188580e+07,0.12679,0.405946,27.915665,1.320514
3,Model 2: Province + Modified Framework,275,3.986219e+07,0.12679,0.392459,26.566988,1.349423


## Key Findings and Recommendations

**Findings (from the n=279 run):**

- **Ranking by total coverage gain:** Model 4 (National + Simple Distance,
  +29.13pp, 41.81% total PCR) > Model 1 (National + Modified, +28.44pp) >
  Model 3 (Province + Simple, +27.92pp) > Model 2 (Province + Modified,
  +26.57pp).
- **National scope beats province scope** in both eligibility settings
  (Model 4 vs 3: +1.22pp; Model 1 vs 2: +1.87pp) -- expected, since an
  unconstrained national solve can concentrate stations wherever marginal
  coverage is highest, while province-level solves must spread across every
  province regardless of local payoff.
- **Simple Distance slightly outperforms Modified Framework** at both
  scopes (National: +0.69pp; Province: +1.35pp), which is expected in
  principle -- restricting to built-up land can only shrink the candidate
  pool -- but the *gap is smaller than the methodology's framing would
  suggest.*
- **Province-level models under-filled their budget:** Model 2 placed only
  275/279 stations and Model 3 placed 277/279 -- at least one province ran
  out of usable candidate sites within its allotted sub-budget. Worth
  identifying which province(s) hit this ceiling before finalizing
  province-level recommendations.
- **MARR sits in a narrow band (1.28-1.35)** across all four models --
  already-covered points are served by roughly 2.3 stations on average
  regardless of approach, with Modified Framework models showing slightly
  more redundancy than their Simple Distance counterparts.

**Caveat that affects the Modified-vs-Simple comparison specifically:** the
`builtup_frac` values in the site tables (e.g. 42.0, 948.0) are far outside
a 0-1 range, while `BUILTUP_THRESHOLD = 0.2` was written assuming a 0-1
scale. That means nearly every candidate site clears the eligibility bar
regardless of the threshold -- likely why Modified Framework and Simple
Distance results are so close. **Recommend rechecking the GHSL raster's
actual units/scale before treating the small Modified-vs-Simple gap as a
real finding**, since it may currently be an artifact of a threshold-units
mismatch rather than genuine evidence that built-up-land restriction barely
matters.

**Recommendations:**

1. If equitable provincial presence matters more than raw efficiency,
   prefer Model 2 or 3 despite their lower total coverage gain.
2. If maximizing total newly-covered population is the sole priority, Model
   4 wins outright -- but confirm the built-up-threshold scale issue above
   before using this to argue eligibility constraints "don't matter much."
3. Investigate the specific province(s) that hit the candidate-capacity
   ceiling in Models 2/3 (compare `province_budgets_modified` /
   `province_budgets_simple` against `capacity_modified` / `capacity_simple`
   to find the shortfall) before presenting province-level results as final.

## Shared paths for Questions 2 & 3

Both sections below were originally standalone notebooks that assumed they
lived two folders deep (`../../data/...`). Since everything is now one
notebook at the project root, they reuse the same `DATA_DIR` / `RESULTS_DIR`
Path objects Q1's Config cell already defined above, rather than redefining
their own relative paths.


In [20]:
GROUP2_ABT_CSV = DATA_DIR / "Group2_ABT.csv"   # shared raw input for Q2 & Q3

RESULTS_Q2_DIR = RESULTS_DIR / "q2"
RESULTS_Q2_DIR.mkdir(parents=True, exist_ok=True)


# Question 2 -- PM2.5 Forecasting (Hourly Resample, Feature Engineering, Modeling)

Paths below have been repointed from the original `../../data/...` /
bare-filename references to the shared `DATA_DIR` / `RESULTS_Q2_DIR` Path
objects defined above. Variable names below (`df`, `hourly`, `results`,
`g`, etc.) are local to this section -- they're freshly reassigned here and
don't carry over any meaning from Question 1's identically-named variables.


In [21]:
import pandas as pd
import numpy as np
import json
import joblib
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import optuna
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, average_precision_score,
    roc_auc_score, confusion_matrix,
)

/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
ABT_DIR = str(GROUP2_ABT_CSV)

df = pd.read_csv(ABT_DIR)
df.head()

/var/folders/5w/2zqvdwxd2nq_f6vfhvcfy4fw0000gn/T/ipykernel_13591/1920498178.py:3: DtypeWarning: Columns (0: area_in_sq.km) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ABT_DIR)


,station,lat,datetime_utc,pm25 in µg/m^3,lon,city,province,population,area_in_sq.km,density_persons/sqkm
0,313_Cubao,14.6227,2025-09-30 07:00:00+00:00,19.652000,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"
1,313_Cubao,14.6227,2025-09-30 22:00:00+00:00,52.500000,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"
2,313_Cubao,14.6227,2025-09-30 23:00:00+00:00,41.965681,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"
3,313_Cubao,14.6227,2025-10-01 00:00:00+00:00,30.948667,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"
4,313_Cubao,14.6227,2025-10-01 01:00:00+00:00,30.113639,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"


### Dropping columns that won't needed

In [23]:
col_to_drop = ['lat', 'lon', 'province', 'population', 'area_in_sq.km']
df.drop(columns=col_to_drop, inplace=True)

### columns renaming

In [24]:
df = df.rename(columns={
    "pm25 in µg/m^3": "pm25_ug_m3",
    "density_persons/sqkm": "density",
})

## PARSE AND SORT

In [25]:
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True, errors="coerce")
df = df.dropna(subset=["datetime_utc", "pm25_ug_m3"])
df = df.sort_values(["station", "datetime_utc"])

## Per-station hourly resample (mean, not sum)
The data has no fixed interval, this code bins from 10am to 10:59am as 10am, and the rest is so on and so forth. Additionally, filled in the jumps of hours(e.g. from 5 jumped to 8 will now includ 6 and 7) but the station, density, and pm2.5 is NaN. The stations and city is retained, but the pm2.5 is averaged. 

reference: Robust prediction of hourly PM2.5 from meteorological data using LightGBM - PMC (https://pmc.ncbi.nlm.nih.gov/articles/PMC8566180/)

In [26]:
hourly = (df.set_index("datetime_utc")
            .groupby("station")
            .resample("1h", origin="epoch")
            .agg(pm25_ug_m3=("pm25_ug_m3", "mean"),
                 city=("city", "first"),
                 density=("density", "first"))
            .reset_index())

## Filling in the missing gaps

Since the jumped hours has been filled in but has missing values, we used linear interpolation and ffill() and bfill() for station and density as they do not change.

In [27]:
hourly = hourly.sort_values(["station", "datetime_utc"]).copy()

hourly["city"]    = hourly.groupby("station")["city"].ffill().bfill()
hourly["density"] = hourly.groupby("station")["density"].ffill().bfill()

# If a station has ALL city/density missing, groupby ffill can't help.
# Fallback: fill with the global mode / median so nothing is NaN.

# Convert density from string-with-commas to numeric
hourly["density"] = (
    hourly["density"]
        .astype(str)                     # ensure it's string (in case of mixed types)
        .str.replace(",", "", regex=False)  # remove thousands separators
        .str.strip()                     # remove stray whitespace
        .replace({"": None, "nan": None, "None": None, "N/A": None})
        .pipe(pd.to_numeric, errors="coerce")  # convert to float, bad → NaN
)

hourly["city"]    = hourly["city"].fillna(hourly["city"].mode().iloc[0])
hourly["density"] = hourly["density"].fillna(hourly["density"].median())

## Selective interpolation for SHORT gaps only
Only fills gaps up to 3 hours, only between real observations.
Long gaps stay NaN, honest.

In [28]:
def interp_short_gaps(s, limit=3):
    """Interpolate only interior gaps <= limit hours."""
    return s.interpolate(method="linear", limit=limit, limit_area="inside")

hourly["pm25_ug_m3"] = (
    hourly.groupby("station")["pm25_ug_m3"]
          .transform(interp_short_gaps)
)

# Recompute the missing flag AFTER interpolation so it reflects
# what is *still* missing in the model input.
hourly["pm25_missing"] = hourly["pm25_ug_m3"].isna().astype(int)

In [29]:
hourly.to_csv(str(DATA_DIR / "Group2_ABT_hourly.csv"), index=False)

## Transformation


### Temporal and Calendar Features
Encodes cyclical time components (hour, day of week, month) using sine/cosine transformations. Raw integer time values (e.g., hour=23 and hour=0) appear far apart to a model even though they are adjacent in reality; sinusoidal encoding preserves the natural periodicity of diurnal, weekly, and seasonal cycles. 

ref: https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0338392

> **Check this before trusting downstream results:** the cell below reads
> `ABT_DIR`, which points at the *raw* `Group2_ABT.csv` -- not the
> hourly-resampled, gap-filled file (`Group2_ABT_hourly.csv`) that was just
> written above. If that's intentional (e.g. feature engineering is meant to
> start fresh from raw data), no action needed. If it's meant to build on the
> resample/interpolation work above, change the read below to
> `pd.read_csv(str(DATA_DIR / "Group2_ABT_hourly.csv"), parse_dates=["datetime_utc"])`.


In [30]:
hourly = pd.read_csv(str(DATA_DIR / "Group2_ABT_hourly.csv"), parse_dates=["datetime_utc"])

dt = hourly["datetime_utc"].dt

# --- Cyclical encoding (keeps hour 23 adjacent to hour 0) ---
hourly["hour_sin"]  = np.sin(2 * np.pi * dt.hour / 24)
hourly["hour_cos"]  = np.cos(2 * np.pi * dt.hour / 24)

hourly["dow_sin"]   = np.sin(2 * np.pi * dt.dayofweek / 7)
hourly["dow_cos"]   = np.cos(2 * np.pi * dt.dayofweek / 7)

hourly["month_sin"] = np.sin(2 * np.pi * (dt.month - 1) / 12)
hourly["month_cos"] = np.cos(2 * np.pi * (dt.month - 1) / 12)

### Lag Features with Missing Indicators
Desc: Builds lagged PM2.5 values at multiple horizons (1, 3, 6, 12, 24, 48, 72 hours) and attaches a binary missingness flag to each. Lags capture autocorrelation and short-term persistence; the missing indicators let the model distinguish "sensor was down" from "value was zero," which is critical when stations have irregular coverage.

ref: https://arxiv.org/html/2607.07279v3

In [31]:
g = hourly.groupby("station")["pm25_ug_m3"]

for lag in [1, 3, 6, 12, 24, 48, 72]:
    col = f"pm25_lag{lag}"
    hourly[col] = g.transform(lambda s, l=lag: s.shift(l))
    hourly[f"{col}_missing"] = hourly[col].isna().astype(int)

## Derivative Features
Desc: Computes rate-of-change features that capture the velocity and acceleration of PM2.5 concentration. These detect transient pollution events (e.g., sudden AQI spikes) that raw values alone miss. Ratios compare the current value to recent baselines, revealing whether air quality is deteriorating relative to normal conditions.

ref: https://arxiv.org/html/2607.07279v3

In [32]:
g = hourly.groupby("station")["pm25_ug_m3"]

hourly["pm25_diff1"]  = g.transform(lambda s: s.diff(1))
hourly["pm25_diff24"] = g.transform(lambda s: s.diff(24))
hourly["pm25_accel"]  = hourly.groupby("station")["pm25_diff1"].transform(
    lambda s: s.diff(1)
)

# Ratios — compute the rolling means needed for these inline
roll24  = g.transform(lambda s: s.rolling(24,  min_periods=18).mean())
roll168 = g.transform(lambda s: s.rolling(168, min_periods=84).mean())

hourly["pm25_ratio_1h_24h"] = hourly["pm25_lag1"] / roll24
hourly["pm25_ratio_24h_7d"] = roll24 / roll168

## Rolling Statistics (supporting features)
Desc: Rolling mean, standard deviation, min, and max over multiple time windows. These summarize recent behavior of the PM2.5 series and are required inputs for the derivative ratio features above. min_periods prevents fake averages across long gaps.

ref: https://arxiv.org/html/2607.07279v3

In [33]:
g = hourly.groupby("station")["pm25_ug_m3"]

for w in [3, 6, 12, 24, 72, 168]:
    mp = max(2, w // 2)  # require at least half the window
    hourly[f"pm25_roll{w}_mean"] = g.transform(
        lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).mean()
    )
    hourly[f"pm25_roll{w}_std"] = g.transform(
        lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).std()
    )
    hourly[f"pm25_roll{w}_max"] = g.transform(
        lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).max()
    )
    hourly[f"pm25_roll{w}_min"] = g.transform(
        lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).min()
    )

## Station & City Context Features
Desc: Adds per-station and per-city summary statistics (mean, std, z-score, deviation). A global model pooling stations of unequal length needs context to know whether a given hourly value is "normal for this station" or "unusually high." The z-score and deviation features encode this directly.

ref: https://pmc.ncbi.nlm.nih.gov/articles/PMC8566180/

In [34]:
# Station-level aggregates
station_stats = (
    hourly.groupby("station")["pm25_ug_m3"]
          .agg(station_mean="mean",
               station_std="std")
)
hourly = hourly.merge(station_stats, on="station", how="left")

hourly["pm25_dev_from_station_mean"] = (
    hourly["pm25_ug_m3"] - hourly["station_mean"]
)
hourly["pm25_zscore"] = (
    (hourly["pm25_ug_m3"] - hourly["station_mean"]) / hourly["station_std"]
)

# City-level aggregates (mean of all stations in city per hour)
city_stats = (
    hourly.groupby(["city", "datetime_utc"])["pm25_ug_m3"]
          .mean()
          .rename("city_pm25_mean")
          .reset_index()
)
hourly = hourly.merge(city_stats, on=["city", "datetime_utc"], how="left")

hourly["pm25_dev_from_city"] = hourly["pm25_ug_m3"] - hourly["city_pm25_mean"]

##  Missingness Summary Features
Desc: Rolling averages of the missingness flag. High missingness often correlates with sensor outages, power failures during storms, or extreme events — informative signals the model can exploit. Complements the per-lag missing indicators from section 2.

ref: https://arxiv.org/html/2607.07279v3

In [35]:
g = hourly.groupby("station")["pm25_missing"]

hourly["missing_roll24"]  = g.transform(
    lambda s: s.rolling(24, min_periods=1).mean()
)
hourly["missing_roll168"] = g.transform(
    lambda s: s.rolling(168, min_periods=1).mean()
)

### THRESHOLD, LABELING

In [36]:
THRESHOLD = 35   # DENR AO

# STEP 1 — Ensure pm25_24h exists (backward-looking 24h mean)
if "pm25_24h" not in hourly.columns:
    g = hourly.groupby("station")["pm25_ug_m3"]
    hourly["pm25_24h"] = g.transform(
        lambda s: s.rolling(24, min_periods=18).mean()
    )


# STEP 2 — Ensure pm25_next24h exists (forward-looking 24h mean)
if "pm25_next24h" not in hourly.columns:
    g = hourly.groupby("station")["pm25_ug_m3"]
    hourly["pm25_next24h"] = g.transform(
        lambda s: s.rolling(24, min_periods=18).mean().shift(-24)
    )


# STEP 3 — Nowcast label (backward-looking, NaN-preserving)
alert_current = pd.Series(np.nan, index=hourly.index, dtype="float64")
mask_cur = hourly["pm25_24h"].notna()
alert_current.loc[mask_cur] = (
    hourly.loc[mask_cur, "pm25_24h"] > THRESHOLD
).astype(float)

hourly["alert_current_24h"] = alert_current.astype("Int64")


# STEP 4 — Preemptive label (forward-looking, NaN-preserving)
alert_next = pd.Series(np.nan, index=hourly.index, dtype="float64")
mask_next = hourly["pm25_next24h"].notna()
alert_next.loc[mask_next] = (
    hourly.loc[mask_next, "pm25_next24h"] > THRESHOLD
).astype(float)

hourly["alert_next24h"] = alert_next.astype("Int64")

print("=== pm25_24h / pm25_next24h ===")
print("pm25_24h NaN:     ", hourly["pm25_24h"].isna().sum())
print("pm25_next24h NaN: ", hourly["pm25_next24h"].isna().sum())

print("\n=== alert labels ===")
for col in ["alert_current_24h", "alert_next24h"]:
    print(f"\n{col}")
    print("  dtype:  ", hourly[col].dtype)
    print("  NaN:    ", hourly[col].isna().sum())
    print("  rate:   ", round(hourly[col].dropna().mean(), 4))
    print("  counts: ")
    print(hourly[col].value_counts(dropna=False))

=== pm25_24h / pm25_next24h ===
pm25_24h NaN:      57665
pm25_next24h NaN:  58199

=== alert labels ===

alert_current_24h
  dtype:   Int64
  NaN:     57665
  rate:    0.0773
  counts: 
alert_current_24h
0       395044
<NA>     57665
1        33072
Name: count, dtype: Int64

alert_next24h
  dtype:   Int64
  NaN:     58199
  rate:    0.0772
  counts: 
alert_next24h
0       394558
<NA>     58199
1        33024
Name: count, dtype: Int64


## Fixing leaky columns
station_mean, station_std, city_pm25_mean, pm25_zscore, pm25_dev_from_station_mean, pm25_dev_from_city were computed using all rows — including those that will become the test set. This leaks test information into training.

Fix: drop these columns from hourly, and recompute them inside your train/test pipeline.

In [37]:
leaky_cols = [
    "station_mean", "station_std",
    "pm25_dev_from_station_mean", "pm25_zscore",
    "city_pm25_mean", "pm25_dev_from_city",
]

hourly = hourly.drop(columns=leaky_cols)

In [38]:
hourly.to_csv(str(DATA_DIR / "Group2_ABT_modeling.csv"), index=False)

In [39]:
hourly = pd.read_csv(str(DATA_DIR / "Group2_ABT_modeling.csv"))

## MODELING

#### Step 1 — Imports and seed

In [40]:
import json
import joblib
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, average_precision_score,
    roc_auc_score, confusion_matrix,
)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
print("Imports OK")

Imports OK


#### Step 2 — Load CSV and repair dtypes

In [41]:
# ---------- 1. Parse datetime back to UTC ----------
hourly["datetime_utc"] = pd.to_datetime(
    hourly["datetime_utc"], utc=True, errors="coerce"
)

# ---------- 2. Cast alert labels back to nullable Int64 ----------
for c in ["alert_current_24h", "alert_next24h"]:
    # First to float (in case it's object), then to Int64 (preserves NaN)
    hourly[c] = pd.to_numeric(hourly[c], errors="coerce").astype("Int64")

# ---------- 3. Verify ----------
print("datetime_utc :", hourly["datetime_utc"].dtype)
print("alert_current_24h:", hourly["alert_current_24h"].dtype,
      "| NaN:", hourly["alert_current_24h"].isna().sum())
print("alert_next24h:", hourly["alert_next24h"].dtype,
      "| NaN:", hourly["alert_next24h"].isna().sum())

print("\nValue counts for alert_current_24h:")
print(hourly["alert_current_24h"].value_counts(dropna=False))

print("\nValue counts for alert_next24h:")
print(hourly["alert_next24h"].value_counts(dropna=False))

datetime_utc : datetime64[us, UTC]
alert_current_24h: Int64 | NaN: 57665
alert_next24h: Int64 | NaN: 58199

Value counts for alert_current_24h:
alert_current_24h
0       395044
<NA>     57665
1        33072
Name: count, dtype: Int64

Value counts for alert_next24h:
alert_next24h
0       394558
<NA>     58199
1        33024
Name: count, dtype: Int64


#### Step 3 — Time-based split

In [42]:
hourly = hourly.sort_values("datetime_utc").reset_index(drop=True)

cutoff = hourly["datetime_utc"].quantile(0.8)
train = hourly[hourly["datetime_utc"] <  cutoff].copy()
test  = hourly[hourly["datetime_utc"] >= cutoff].copy()

print("Train rows:", len(train),
      "| range:", train["datetime_utc"].min(), "→", train["datetime_utc"].max())
print("Test  rows:", len(test),
      "| range:", test["datetime_utc"].min(),  "→", test["datetime_utc"].max())

Train rows: 388622 | range: 2023-09-06 20:00:00+00:00 → 2026-03-12 18:00:00+00:00
Test  rows: 97159 | range: 2026-03-12 19:00:00+00:00 → 2026-09-04 07:00:00+00:00


#### Step 4 — Recompute context features on train only

In [43]:
# ---------- Station-level stats (fit on train) ----------
station_stats = (
    train.groupby("station")["pm25_ug_m3"]
         .agg(station_mean="mean", station_std="std")
         .reset_index()
)
train = train.merge(station_stats, on="station", how="left")
test  = test.merge(station_stats,  on="station", how="left")

for df in (train, test):
    df["station_std"] = df["station_std"].fillna(0)
    df["pm25_dev_from_station_mean"] = df["pm25_ug_m3"] - df["station_mean"]
    df["pm25_zscore"] = (
        (df["pm25_ug_m3"] - df["station_mean"]) / df["station_std"]
    ).fillna(0)

# ---------- City-level stats (fit on train) ----------
city_stats = (
    train.groupby(["city", "datetime_utc"])["pm25_ug_m3"]
         .mean().rename("city_pm25_mean").reset_index()
)
train = train.merge(city_stats, on=["city", "datetime_utc"], how="left")
test  = test.merge(city_stats,  on=["city", "datetime_utc"], how="left")

for df in (train, test):
    df["pm25_dev_from_city"] = df["pm25_ug_m3"] - df["city_pm25_mean"]

print("Context features added.")
print("train shape:", train.shape, "| test shape:", test.shape)

Context features added.
train shape: (388622, 67) | test shape: (97159, 67)


#### Step 5 — Define feature list

In [44]:
exclude_from_features = {
    # identifiers
    "datetime_utc", "city",
    # labels
    "alert_current_24h", "alert_next24h",
    # label source windows
    "pm25_24h", "pm25_24h_std", "pm25_next24h",
    # raw current value
    "pm25_ug_m3",
    # 24h-window features (residual leak)
    "pm25_roll24_mean", "pm25_roll24_std",
    "pm25_roll24_max",  "pm25_roll24_min",
    # 12h-window features
    "pm25_roll12_mean", "pm25_roll12_std",
    "pm25_roll12_max",  "pm25_roll12_min",
    # ratios on the 24h window
    "pm25_ratio_1h_24h", "pm25_ratio_24h_7d",
}

features = [c for c in train.columns if c not in exclude_from_features]

features = [c for c in train.columns if c not in exclude_from_features]
print("n features:", len(features))
print(features)

# Persist for reproducibility
with open(str(RESULTS_Q2_DIR / "feature_list.json"), "w") as f:
    json.dump(features, f, indent=2)

n features: 50
['station', 'density', 'pm25_missing', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'pm25_lag1', 'pm25_lag1_missing', 'pm25_lag3', 'pm25_lag3_missing', 'pm25_lag6', 'pm25_lag6_missing', 'pm25_lag12', 'pm25_lag12_missing', 'pm25_lag24', 'pm25_lag24_missing', 'pm25_lag48', 'pm25_lag48_missing', 'pm25_lag72', 'pm25_lag72_missing', 'pm25_diff1', 'pm25_diff24', 'pm25_accel', 'pm25_roll3_mean', 'pm25_roll3_std', 'pm25_roll3_max', 'pm25_roll3_min', 'pm25_roll6_mean', 'pm25_roll6_std', 'pm25_roll6_max', 'pm25_roll6_min', 'pm25_roll72_mean', 'pm25_roll72_std', 'pm25_roll72_max', 'pm25_roll72_min', 'pm25_roll168_mean', 'pm25_roll168_std', 'pm25_roll168_max', 'pm25_roll168_min', 'missing_roll24', 'missing_roll168', 'station_mean', 'station_std', 'pm25_dev_from_station_mean', 'pm25_zscore', 'city_pm25_mean', 'pm25_dev_from_city']


#### Step 6 — Drop NaN targets per model

In [45]:
TARGET_REG = "pm25_next24h"   # ← was "pm25_24h"
TARGET_NOW = "alert_current_24h"
TARGET_NXT = "alert_next24h"

# Regression — target = pm25_next24h (forward-looking)
train_reg = train.dropna(subset=["pm25_next24h"]).copy()
test_reg  = test.dropna(subset=["pm25_next24h"]).copy()

# Nowcast classifier — unchanged (backward target is its nature)
train_now = train.dropna(subset=["alert_current_24h"]).copy()
test_now  = test.dropna(subset=["alert_current_24h"]).copy()

# Preemptive classifier — unchanged
train_nxt = train.dropna(subset=["alert_next24h"]).copy()
test_nxt  = test.dropna(subset=["alert_next24h"]).copy()

print("Regression  train/test:", len(train_reg), len(test_reg))
print("Nowcast     train/test:", len(train_now), len(test_now))
print("Preemptive  train/test:", len(train_nxt), len(test_nxt))

Regression  train/test: 345457 82125
Nowcast     train/test: 344979 83137
Preemptive  train/test: 345457 82125


In [46]:
# ---- Cast station to category once, for all downstream models ----
cats = sorted(train_reg["station"].astype(str).unique())

train_reg["station"] = pd.Categorical(train_reg["station"].astype(str), categories=cats)
test_reg["station"]  = pd.Categorical(test_reg["station"].astype(str),  categories=cats)

train_now["station"] = pd.Categorical(train_now["station"].astype(str), categories=cats)
test_now["station"]  = pd.Categorical(test_now["station"].astype(str),  categories=cats)

train_nxt["station"] = pd.Categorical(train_nxt["station"].astype(str), categories=cats)
test_nxt["station"]  = pd.Categorical(test_nxt["station"].astype(str),  categories=cats)

print("station dtype:", train_reg["station"].dtype)
print("n categories: ", len(cats))

station dtype: category
n categories:  71


/var/folders/5w/2zqvdwxd2nq_f6vfhvcfy4fw0000gn/T/ipykernel_13591/1302535563.py:5: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  test_reg["station"]  = pd.Categorical(test_reg["station"].astype(str),  categories=cats)
/var/folders/5w/2zqvdwxd2nq_f6vfhvcfy4fw0000gn/T/ipykernel_13591/1302535563.py:8: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  test_now["station"]  = pd.Categorical(test_now["station"].astype(str),  categories=cats)
/var/folders/5w/2zqvdwxd2nq_f6vfhvcfy4fw0000gn/T/ipykernel_13591/1302535563.py:11: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  test_nxt["station"]  = pd.Categorical(tes

##### Step 6a — Ridge tuning

In [47]:
# ---- Rebuild one-hot features (as before) ----
train_ridge = pd.get_dummies(
    train_reg[features], columns=["station"], prefix="st", dummy_na=False
)
test_ridge = pd.get_dummies(
    test_reg[features],  columns=["station"], prefix="st", dummy_na=False
)
train_ridge, test_ridge = train_ridge.align(
    test_ridge, join="left", axis=1, fill_value=0
)

# ---- Pipeline ----
ridge_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
    ("model",  Ridge(random_state=SEED)),
])

# ---- Time-series cross-validation (no shuffling) ----
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "model__alpha": [0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000],
}

ridge_grid = GridSearchCV(
    ridge_pipe,
    param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)
ridge_grid.fit(train_ridge, train_reg[TARGET_REG])

print("Best alpha:", ridge_grid.best_params_)
print("Best CV RMSE:", -ridge_grid.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best alpha: {'model__alpha': 1000}
Best CV RMSE: 6.615164439816487


##### Step 6b — LightGBM tuning (Optuna)

In [48]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---- Define inner validation slice ----
val_cutoff = train_reg["datetime_utc"].quantile(0.8)
train_fit  = train_reg[train_reg["datetime_utc"] <  val_cutoff].copy()
train_val  = train_reg[train_reg["datetime_utc"] >= val_cutoff].copy()

# ---- Cast station to category HERE, on the slices actually used ----
cats = sorted(train_reg["station"].astype(str).unique())
for df in (train_fit, train_val):
    df["station"] = pd.Categorical(df["station"].astype(str), categories=cats)

# ---- Sanity check ----
bad = [c for c in features if train_fit[c].dtype == "object"]
print("fit rows:", len(train_fit), "| val rows:", len(train_val))
print("object-dtype features:", bad)   # must be []


# ---- Objective ----
def objective(trial):
    params = {
        "n_estimators":      5000,
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "random_state":      SEED,
        "n_jobs":            -1,
        "verbosity":         -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        train_fit[features], train_fit[TARGET_REG],
        eval_set=[(train_val[features], train_val[TARGET_REG])],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    pred = model.predict(train_val[features])
    return np.sqrt(mean_squared_error(train_val[TARGET_REG], pred))

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("Best val RMSE:", study.best_value)

best_lgbm_params = study.best_params

fit rows: 276347 | val rows: 69110
object-dtype features: []


  0%|          | 0/30 [00:00<?, ?it/s]/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 7.12463:   3%|▎         | 1/30 [00:13<06:26, 13.31s/it]/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Best trial: 0. Best value: 7.12463:   7%|▋         | 2/30 [00:17<03:40,  7.89s/it]/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'ev

Best params:
  learning_rate: 0.034149258537347314
  num_leaves: 24
  min_child_samples: 36
  subsample: 0.60484642300241
  colsample_bytree: 0.6443635290662082
  reg_alpha: 0.0034177263133857057
  reg_lambda: 0.779395265785988
Best val RMSE: 7.023782670592707


#### Step 7 — Global LightGBM (regression, champion)

In [49]:
# ---- Cast station ----
cats = sorted(train_reg["station"].astype(str).unique())
train_reg["station"] = pd.Categorical(train_reg["station"].astype(str), categories=cats)
test_reg["station"]  = pd.Categorical(test_reg["station"].astype(str),  categories=cats)

# ---- Inner validation slice ----
val_cutoff = train_reg["datetime_utc"].quantile(0.8)
train_fit  = train_reg[train_reg["datetime_utc"] <  val_cutoff]
train_val  = train_reg[train_reg["datetime_utc"] >= val_cutoff]

# ---- Final model with tuned params ----
model_lgbm = lgb.LGBMRegressor(
    n_estimators=5000,
    **best_lgbm_params,          # ← tuned hyperparameters
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

model_lgbm.fit(
    train_fit[features], train_fit[TARGET_REG],
    eval_set=[(train_val[features], train_val[TARGET_REG])],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=200),
    ],
)
print("Best iteration:", model_lgbm.best_iteration_)

# ---- Evaluate on held-out test ----
pred = model_lgbm.predict(test_reg[features])
y    = test_reg[TARGET_REG].to_numpy()

rmse = np.sqrt(mean_squared_error(y, pred))
mae  = mean_absolute_error(y, pred)
r2   = r2_score(y, pred)

print("\n=== Global LightGBM (tuned) ===")
print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")
print(f"R²   : {r2:.4f}")

# ---- Fair baseline comparison ----
mask = test_reg["pm25_lag24"].notna()
y_subset    = test_reg.loc[mask, TARGET_REG].to_numpy()
pred_subset = pred[mask]
base_subset = test_reg.loc[mask, "pm25_lag24"].to_numpy()

rmse_model_sub = np.sqrt(mean_squared_error(y_subset, pred_subset))
rmse_base_sub  = np.sqrt(mean_squared_error(y_subset, base_subset))

print(f"\nFair comparison on {mask.sum():,} rows:")
print(f"Model    → RMSE: {rmse_model_sub:.3f}")
print(f"Baseline → RMSE: {rmse_base_sub:.3f}")
print(f"Improvement: {(1 - rmse_model_sub / rmse_base_sub) * 100:.2f}%")

# ---- Feature importance ----
importance = (
    pd.Series(model_lgbm.feature_importances_, index=features)
      .sort_values(ascending=False)
)
print("\nTop 20 features:")
print(importance.head(20))

joblib.dump(model_lgbm, str(RESULTS_Q2_DIR / "global_lgbm_pm25_tuned.pkl"))
print("\nSaved: global_lgbm_pm25_tuned.pkl")

/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[200]	valid_0's rmse: 7.02794	valid_0's l2: 49.3919
Best iteration: 185

=== Global LightGBM (tuned) ===
RMSE : 9.568
MAE  : 6.398
R²   : 0.4522

Fair comparison on 79,124 rows:
Model    → RMSE: 9.616
Baseline → RMSE: 16.101
Improvement: 40.27%

Top 20 features:
station              401
pm25_roll72_min      262
pm25_roll168_min     258
pm25_roll72_mean     207
pm25_roll168_max     195
month_cos            189
month_sin            184
city_pm25_mean       178
pm25_roll168_mean    171
missing_roll168      168
dow_cos              163
pm25_roll72_max      157
dow_sin              138
station_mean         132
pm25_lag12           132
pm25_roll168_std     132
pm25_roll6_min       116
pm25_roll72_std      111
missing_roll24       110
pm25_lag48            81
dtype: int32

Saved: global_lgbm_pm25_tuned.pkl


#### Step 8 — Ridge regression (challenger)

In [50]:
best_alpha = ridge_grid.best_params_["model__alpha"]
print("Using alpha =", best_alpha)

model_ridge = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
    ("model",  Ridge(alpha=best_alpha, random_state=SEED)),
])

model_ridge.fit(train_ridge, train_reg[TARGET_REG])

# ---- Evaluate ----
pred_r = model_ridge.predict(test_ridge)
y_r    = test_reg[TARGET_REG].to_numpy()

rmse_r = np.sqrt(mean_squared_error(y_r, pred_r))
mae_r  = mean_absolute_error(y_r, pred_r)
r2_r   = r2_score(y_r, pred_r)

print("\n=== Ridge Regression (tuned) ===")
print(f"RMSE : {rmse_r:.3f}")
print(f"MAE  : {mae_r:.3f}")
print(f"R²   : {r2_r:.4f}")

joblib.dump(model_ridge, str(RESULTS_Q2_DIR / "ridge_pm25_tuned.pkl"))
print("Saved: ridge_pm25_tuned.pkl")

Using alpha = 1000

=== Ridge Regression (tuned) ===
RMSE : 9.282
MAE  : 6.294
R²   : 0.4844
Saved: ridge_pm25_tuned.pkl


#### Step 9 — LightGBM Classifier (Nowcast — alert_current_24h)

In [51]:
# ---- Cast station ----
cats_now = sorted(train_now["station"].astype(str).unique())
train_now["station"] = pd.Categorical(train_now["station"].astype(str), categories=cats_now)
test_now["station"]  = pd.Categorical(test_now["station"].astype(str),  categories=cats_now)

# ---- Inner validation slice ----
val_cutoff_now = train_now["datetime_utc"].quantile(0.8)
train_now_fit  = train_now[train_now["datetime_utc"] <  val_cutoff_now]
train_now_val  = train_now[train_now["datetime_utc"] >= val_cutoff_now]

# ---- Class balance ----
rate_now = train_now_fit[TARGET_NOW].mean()
spw_now  = (1 - rate_now) / rate_now
print(f"Nowcast positive rate: {rate_now:.4f} | scale_pos_weight = {spw_now:.2f}")

# ---- Model (tuned) ----
clf_now = lgb.LGBMClassifier(
    n_estimators=5000,
    **best_lgbm_params,           # ← tuned hyperparameters from Step 6b
    scale_pos_weight=spw_now,     # derived, not tuned
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

clf_now.fit(
    train_now_fit[features], train_now_fit[TARGET_NOW].astype(int),
    eval_set=[(train_now_val[features], train_now_val[TARGET_NOW].astype(int))],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=200),
    ],
)
print("Best iteration:", clf_now.best_iteration_)

# ---- Evaluate ----
proba = clf_now.predict_proba(test_now[features])[:, 1]
pred  = (proba >= 0.5).astype(int)
y_now = test_now[TARGET_NOW].astype(int).to_numpy()

print("\n=== LightGBM Classifier (Nowcast, tuned) ===")
print(classification_report(y_now, pred, digits=3))
print("PR-AUC:", round(average_precision_score(y_now, proba), 4))
print("ROC-AUC:", round(roc_auc_score(y_now, proba), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_now, pred))

joblib.dump(clf_now, str(RESULTS_Q2_DIR / "clf_nowcast_tuned.pkl"))
print("Saved: clf_nowcast_tuned.pkl")

Nowcast positive rate: 0.0646 | scale_pos_weight = 14.47


/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[200]	valid_0's average_precision: 0.888095	valid_0's binary_logloss: 0.0851933
Best iteration: 270

=== LightGBM Classifier (Nowcast, tuned) ===
              precision    recall  f1-score   support

           0      0.989     0.939     0.964     70908
           1      0.728     0.940     0.821     12229

    accuracy                          0.940     83137
   macro avg      0.859     0.940     0.892     83137
weighted avg      0.951     0.940     0.943     83137

PR-AUC: 0.9373
ROC-AUC: 0.9875

Confusion matrix:
[[66615  4293]
 [  734 11495]]
Saved: clf_nowcast_tuned.pkl


#### Step 10 — LightGBM Classifier (Preemptive — alert_next24h)

In [52]:
# ---- Cast station ----
cats_nxt = sorted(train_nxt["station"].astype(str).unique())
train_nxt["station"] = pd.Categorical(train_nxt["station"].astype(str), categories=cats_nxt)
test_nxt["station"]  = pd.Categorical(test_nxt["station"].astype(str),  categories=cats_nxt)

# ---- Inner validation slice ----
val_cutoff_nxt = train_nxt["datetime_utc"].quantile(0.8)
train_nxt_fit  = train_nxt[train_nxt["datetime_utc"] <  val_cutoff_nxt]
train_nxt_val  = train_nxt[train_nxt["datetime_utc"] >= val_cutoff_nxt]

# ---- Class balance ----
rate_nxt = train_nxt_fit[TARGET_NXT].mean()
spw_nxt  = (1 - rate_nxt) / rate_nxt
print(f"Preemptive positive rate: {rate_nxt:.4f} | scale_pos_weight = {spw_nxt:.2f}")

# ---- Model (tuned) ----
clf_nxt = lgb.LGBMClassifier(
    n_estimators=5000,
    **best_lgbm_params,           # ← tuned hyperparameters from Step 6b
    scale_pos_weight=spw_nxt,     # derived, not tuned
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

clf_nxt.fit(
    train_nxt_fit[features], train_nxt_fit[TARGET_NXT].astype(int),
    eval_set=[(train_nxt_val[features], train_nxt_val[TARGET_NXT].astype(int))],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=200),
    ],
)
print("Best iteration:", clf_nxt.best_iteration_)

# ---- Evaluate ----
proba_n = clf_nxt.predict_proba(test_nxt[features])[:, 1]
pred_n  = (proba_n >= 0.5).astype(int)
y_nxt   = test_nxt[TARGET_NXT].astype(int).to_numpy()

print("\n=== LightGBM Classifier (Preemptive, tuned) ===")
print(classification_report(y_nxt, pred_n, digits=3))
print("PR-AUC:", round(average_precision_score(y_nxt, proba_n), 4))
print("ROC-AUC:", round(roc_auc_score(y_nxt, proba_n), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_nxt, pred_n))

joblib.dump(clf_nxt, str(RESULTS_Q2_DIR / "clf_preemptive_tuned.pkl"))
print("Saved: clf_preemptive_tuned.pkl")

Preemptive positive rate: 0.0645 | scale_pos_weight = 14.50


/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best iteration: 7

=== LightGBM Classifier (Preemptive, tuned) ===
              precision    recall  f1-score   support

           0      0.851     1.000     0.920     69906
           1      0.000     0.000     0.000     12219

    accuracy                          0.851     82125
   macro avg      0.426     0.500     0.460     82125
weighted avg      0.725     0.851     0.783     82125

PR-AUC: 0.424
ROC-AUC: 0.815

Confusion matrix:
[[69906     0]
 [12219     0]]
Saved: clf_preemptive_tuned.pkl


/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/paulocuyo/PycharmProjects/Prescriptive_Analytics/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

#### Step 11 — Compare all models

In [53]:
results = pd.DataFrame({
    "Model":  ["Global LightGBM", "Ridge", "Nowcast Clf", "Preemptive Clf"],
    "Task":   ["Regression", "Regression", "Classification", "Classification"],
    "RMSE":   [rmse, rmse_r, np.nan, np.nan],
    "MAE":    [mae, mae_r, np.nan, np.nan],
    "R²":     [r2, r2_r, np.nan, np.nan],
    "PR-AUC": [np.nan, np.nan,
               average_precision_score(y_now, proba),
               average_precision_score(y_nxt, proba_n)],
})
print(results.round(4).to_string(index=False))

          Model           Task   RMSE    MAE     R²  PR-AUC
Global LightGBM     Regression 9.5677 6.3985 0.4522     NaN
          Ridge     Regression 9.2818 6.2938 0.4844     NaN
    Nowcast Clf Classification    NaN    NaN    NaN  0.9373
 Preemptive Clf Classification    NaN    NaN    NaN  0.4240


FINDINGS:

* Ridge regression beat Global LightGBM
* Nowcasting is good, Forecasting is hard which is expected since the 
* Published literatures with only regressive features tops out at R^2 0.4-0.6. This findings aligns with ours (0.44–0.48).
* Meteorological features are a must: wind, humidity, temperature, boundary layer, precipitation are used in almost all study. However, the current data none any of these which might be the reason.

#### Conclusion:
A leak-free, time-validated pipeline shows that 24h-ahead PM2.5 is moderately predictable (R² ~0.48) with a dominantly linear signal, that nowcasting exceedances is easy while preemptive prediction remains hard but feasible (PR-AUC ~0.42), and that meteorological features are the most promising next lever to lift forecasting performance.

### Recommendation in using this model output:
* Deploy a current exeedances as it has high result (~92).
* For the next 24hr alert prediction, use only as internal early-warning, not yet for automatic public alert
* For predicting PM 2.5 magnitudes, the regression models predicted with 9-10 units error and an r^2 of ~.48. This should be use only for severity grading, not a precise numeric forecasts.

---
# Question 3 -- Sensor Health / Anomaly Detection

Output directories and the shared input CSV path below have been repointed
to the shared `DATA_DIR` / `RESULTS_DIR` Path objects defined earlier,
instead of the original `../../data/...` relative paths. Two output writes
that originally used a bare `src/analysis` (current-directory) path have also been
redirected into `OUT_PROOF` for consistency with every other Q3 output.


# **Analytical Question 3**

#### **Package Installation and Directories**

In [54]:
import sys
import subprocess

def auto_install(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"{package_name} is missing! Installing it now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"{package_name} successfully installed.")

auto_install("tensorflow")
auto_install("scikit-learn", import_name="sklearn")
auto_install("scipy")
auto_install("pyod")
auto_install("pandas")

In [55]:
import numpy as np
import pandas as pd
import os
import json
import time
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import interp1d
from pyod.models.ecod import ECOD

In [56]:
# For Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Output Directories
OUT_PROOF = str(RESULTS_DIR / "outputs_Q3");  os.makedirs(OUT_PROOF, exist_ok=True)
OUT_M1    = str(RESULTS_DIR / "outputs_Q3" / "outputs_model1"); os.makedirs(OUT_M1,    exist_ok=True)
OUT_M2    = str(RESULTS_DIR / "outputs_Q3" / "outputs_model2"); os.makedirs(OUT_M2,    exist_ok=True)
OUT_M3    = str(RESULTS_DIR / "outputs_Q3" / "outputs_model3"); os.makedirs(OUT_M3,    exist_ok=True)

#### **Load Dataset**

In [57]:
CSV_PATH = str(GROUP2_ABT_CSV)

df = pd.read_csv(CSV_PATH, on_bad_lines="skip", low_memory=False)
pm_col = [c for c in df.columns if "pm25 in µg/m^3" in c.lower()][0]

# clean ["datetime_utc"] and ["station"]
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], errors="coerce")
df = df.dropna(subset=["datetime_utc", "station"])
df[pm_col] = pd.to_numeric(df[pm_col], errors="coerce")
df = df.sort_values(["station", "datetime_utc"]).reset_index(drop=True)

# clean ["density_persons/sqkm"]
if "density_persons/sqkm" in df.columns:
    df["density_persons/sqkm"] = (
        df["density_persons/sqkm"].astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

print(f"Loaded {len(df):,} rows from {df["station"].nunique()} stations")

Loaded 1,027,171 rows from 97 stations


#### **Proofs and Evidences**

##### **Time Gaps Per Station**

In [58]:
df["gap_min"] = df.groupby("station")["datetime_utc"].diff().dt.total_seconds()/60
gaps = df.dropna(subset=["gap_min"])

station_stats = gaps.groupby("station")["gap_min"].agg(
    n_records = "count",
    median_gap_min = "median",
    p75_gap_min = lambda x: x.quantile(0.75),
    p90_gap_min = lambda x: x.quantile(0.90),
    p95_gap_min = lambda x: x.quantile(0.95),
    mean_gap_min = "mean",
).round(2)

station_stats.to_csv(f"{OUT_PROOF}/TimeGaps.csv")
print("====== TIME GAPS PER STATION ======")
display(station_stats.describe().round(2))

====== TIME GAPS PER STATION ======


,n_records,median_gap_min,p75_gap_min,p90_gap_min,p95_gap_min,mean_gap_min
count,96.00,96.00,96.00,96.00,96.00,96.00
mean,10698.69,30.72,33.00,54.53,116.77,89.28
std,6983.14,23.13,33.47,120.36,245.52,237.22
min,1.00,5.58,5.60,5.60,5.82,7.51
25%,2497.25,11.30,14.64,17.09,60.00,23.83
50%,13642.00,17.33,17.38,17.48,60.29,24.80
75%,16999.00,60.00,60.00,60.00,69.04,64.33
max,16999.00,60.00,270.00,810.00,1458.00,1783.36


##### **Median Gaps per station across different time targets**
*What percentage of our stations are reporting data at or faster than X minutes?*

In [59]:
print("====== Share of Stations at Candidate Thresholds ======")
for thr in [5, 10, 15, 20, 30, 40, 50, 60]:
    share = (station_stats["median_gap_min"] <= thr).mean()
    print(f"  Median Gap <= {thr:>3} min: {share:6.2%} of stations")

====== Share of Stations at Candidate Thresholds ======
  Median Gap <=   5 min:  0.00% of stations
  Median Gap <=  10 min: 25.00% of stations
  Median Gap <=  15 min: 26.04% of stations
  Median Gap <=  20 min: 62.50% of stations
  Median Gap <=  30 min: 62.50% of stations
  Median Gap <=  40 min: 62.50% of stations
  Median Gap <=  50 min: 62.50% of stations
  Median Gap <=  60 min: 100.00% of stations


#### **Per Station Coverage: 20 minutes vs 1 hour**

In [60]:
print("=" * 72)
print("COVERAGE: 20 min vs 1 h  (per-station, own active period)")
print("=" * 72)

CANDIDATES = ['20min', '1h']

# ---- Step 1: classify stations by native cadence cluster ----
def cadence_cluster(median_gap):
    if median_gap <= 10:
        return 'A_high_freq'      # ~5.7–8.0 min
    if median_gap <= 20:
        return 'B_candidate'     # ~15-20 min
    return 'C_hourly'             # 60 min

station_stats['cluster'] = station_stats['median_gap_min'].apply(cadence_cluster)

cluster_sizes = station_stats['cluster'].value_counts()
print("\nCluster sizes:")
for cl in ['A_high_freq', 'B_candidate', 'C_hourly']:
    if cl not in cluster_sizes: continue
    n = cluster_sizes[cl]
    print(f"  {cl:>15}: {n:>3} stations  ({n/len(station_stats):>5.1%})")

# ---- Step 2: per-station coverage on own active period ----
coverage_records = []
for station, group in df.groupby('station'):
    if station not in station_stats.index:
        continue

    cluster = station_stats.loc[station, 'cluster']
    t0, t1 = group['datetime_utc'].min(), group['datetime_utc'].max()
    g = group.set_index('datetime_utc')[pm_col].sort_index()

    row = {
        'station':      station,
        'cluster':      cluster,
        'n_records':    len(g),
        'active_hours': (t1 - t0).total_seconds() / 3600,
    }
    for res in CANDIDATES:
        s = g.resample(res).mean()
        row[f'coverage_{res}'] = float(s.notna().mean())
    coverage_records.append(row)

coverage_df = pd.DataFrame(coverage_records).set_index('station')

# ---- Step 3: mean coverage per cluster ----
print("\n" + "-" * 72)
print("Mean coverage per cluster")
print("-" * 72)
agg = coverage_df.groupby('cluster')[[f'coverage_{r}' for r in CANDIDATES]].mean()
print(agg.round(3).to_string())

# ---- Step 4: network-weighted coverage ----
print("\n" + "-" * 72)
print("Network-weighted mean coverage")
print("-" * 72)
weights = cluster_sizes / cluster_sizes.sum()
weighted = (agg.T * weights).T.sum()
for res in CANDIDATES:
    print(f"  {res:>5}: {weighted[f'coverage_{res}']:.3f}")

# ---- Step 5: fraction of stations at >= 0.95 coverage ----
print("\n" + "-" * 72)
print("Fraction of stations with coverage >= 0.95")
print("-" * 72)
for res in CANDIDATES:
    col = f'coverage_{res}'
    n = (coverage_df[col] >= 0.95).sum()
    print(f"  {res:>5}: {n:>3}/{len(coverage_df)} stations ({n/len(coverage_df):.1%})")

# ---- Step 6: which clusters are well-served at each resolution ----
print("\n" + "-" * 72)
print("Stations with coverage >= 0.95, by cluster")
print("-" * 72)
for cl in ['A_high_freq', 'B_candidate', 'C_hourly']:
    subset = coverage_df[coverage_df['cluster'] == cl]
    if len(subset) == 0: continue
    print(f"\n  {cl}  (n = {len(subset)}):")
    for res in CANDIDATES:
        col = f'coverage_{res}'
        n = (subset[col] >= 0.95).sum()
        print(f"    {res:>5}: {n:>3}/{len(subset)} ({n/len(subset):.1%})")

# ---- Step 7: side-by-side per-station table (top 15 by 20-min gap) ----
print("\n" + "-" * 72)
print("Stations where 20 min gains the most over 1 h")
print("-" * 72)
coverage_df['gain_20min_over_1h'] = (
    coverage_df['coverage_20min'] - coverage_df['coverage_1h']
)
top_gain = coverage_df.sort_values('gain_20min_over_1h', ascending=False).head(15)
print(top_gain[['cluster', 'coverage_20min', 'coverage_1h',
                'gain_20min_over_1h']].round(3).to_string())

# ---- Step 8: save ----
coverage_df.to_csv(f'{OUT_PROOF}/coverage_20min_vs_1h_per_station.csv')
agg.to_csv(f'{OUT_PROOF}/coverage_20min_vs_1h_per_cluster.csv')
print(f"\nSaved:")
print(f"  {OUT_PROOF}/coverage_20min_vs_1h_per_station.csv")
print(f"  {OUT_PROOF}/coverage_20min_vs_1h_per_cluster.csv")

COVERAGE: 20 min vs 1 h  (per-station, own active period)

Cluster sizes:
      A_high_freq:  24 stations  (25.0%)
      B_candidate:  36 stations  (37.5%)
         C_hourly:  36 stations  (37.5%)

------------------------------------------------------------------------
Mean coverage per cluster
------------------------------------------------------------------------
             coverage_20min  coverage_1h
cluster                                 
A_high_freq           0.540        0.638
B_candidate           0.717        0.767
C_hourly              0.252        0.732

------------------------------------------------------------------------
Network-weighted mean coverage
------------------------------------------------------------------------
  20min: 0.498
     1h: 0.722

------------------------------------------------------------------------
Fraction of stations with coverage >= 0.95
------------------------------------------------------------------------
  20min:   0/96 stations (0

##### **RMSE: 20 minutes vs 1 hour**

In [61]:
from scipy.interpolate import interp1d

for res in ['20min', '1h']:
    errors = []
    for station, group in df.groupby('station'):
        g = group.set_index('datetime_utc')[pm_col].sort_index()
        coarse = g.resample(res).mean().dropna()
        if len(coarse) < 10: continue
        # Interpolate coarse back to original timestamps
        f = interp1d(coarse.index.astype('int64'), coarse.values,
                     kind='linear', fill_value='extrapolate')
        reconstructed = f(g.index.astype('int64'))
        errors.append(np.sqrt(np.mean((g.values - reconstructed) ** 2)))
    print(f"{res}: mean reconstruction RMSE = {np.mean(errors):.3f}")

20min: mean reconstruction RMSE = 3.731
1h: mean reconstruction RMSE = 4.807


#### **Cadence Cluster**

In [62]:
def cadence_cluster(med):
    if med <= 10:  return 'A_high_freq'
    if med <= 20:  return 'B_candidate'
    return 'C_hourly'

cadence = station_stats.copy()
cadence['cluster']      = cadence['median_gap_min'].apply(cadence_cluster)
cadence['irregularity'] = cadence['p95_gap_min'] / cadence['median_gap_min'].clip(lower=1e-6)

cluster_counts = cadence['cluster'].value_counts()
print("\nCadence clusters:")
for cl in ['A_high_freq', 'B_candidate', 'C_hourly']:
    if cl in cluster_counts:
        n = cluster_counts[cl]
        print(f"    {cl:<16} {n:>3} stations ({n/len(cadence):.1%})")


Cadence clusters:
    A_high_freq       24 stations (25.0%)
    B_candidate       36 stations (37.5%)
    C_hourly          36 stations (37.5%)


#### **Resampling**

In [63]:
RESAMPLE = "20min"

print(f"\nResampling to {RESAMPLE} ...")

df_res = (df.set_index('datetime_utc')
            .groupby('station')
            .resample(RESAMPLE)
            .agg({pm_col: 'mean'})
            .reset_index())

pivot = df_res.pivot(index='datetime_utc', columns='station',
                     values=pm_col).sort_index()

hours_per_bin = pd.Timedelta(RESAMPLE).total_seconds() / 3600.0
print(f"    Grid: {pivot.shape[0]:,} bins × {pivot.shape[1]} stations")
print(f"    Bin width: {hours_per_bin:.3f} h")


Resampling to 20min ...
    Grid: 78,732 bins × 97 stations
    Bin width: 0.333 h


#### **Station Gating**

In [64]:
MIN_N_RECORDS      = 500
MIN_COVERAGE_20MIN = 0.30

print("\nStation gating ...")

cov_rows = []
for st in pivot.columns:
    s = pivot[st].dropna()
    if len(s) < 2:
        cov_rows.append({'station': st, 'n_bins': 0,
                         'active_bins': 0, 'coverage': 0.0})
        continue
    t0, t1 = s.index.min(), s.index.max()
    span = max(1, int((t1 - t0) / pd.Timedelta(RESAMPLE)) + 1)
    cov_rows.append({'station': st, 'n_bins': len(s),
                     'active_bins': span, 'coverage': len(s) / span})

coverage_df = pd.DataFrame(cov_rows).set_index('station')
coverage_df = coverage_df.join(
    cadence[['cluster', 'median_gap_min', 'irregularity']], how='left')

keep = (coverage_df['n_bins'] >= MIN_N_RECORDS) & \
       (coverage_df['coverage'] >= MIN_COVERAGE_20MIN)
excluded = coverage_df[~keep].copy()
excluded['reason'] = np.where(
    coverage_df.loc[~keep, 'n_bins'] < MIN_N_RECORDS, 'low_n_records',
    'low_coverage')

valid_stations = coverage_df.index[keep].tolist()
pivot       = pivot[valid_stations]
coverage_df = coverage_df.loc[valid_stations]

print(f"    Kept     : {len(valid_stations)} / {len(cov_rows)}")
print(f"    Excluded : {len(excluded)}  ({excluded['reason'].value_counts().to_dict()})")
excluded.to_csv(f'{OUT_PROOF}/excluded_stations.csv')


Station gating ...
    Kept     : 67 / 97
    Excluded : 30  ({'low_n_records': 18, 'low_coverage': 12})


#### **Downtime Metrics**

In [65]:
print("\nDowntime metrics (cadence-adjusted, per-station active period) ...")

median_gap = cadence.reindex(pivot.columns)['median_gap_min'].values
expected_coverage = np.minimum(1.0, 20.0 / median_gap)

# Actual coverage measured earlier in the station-gating cell
actual_coverage = coverage_df['coverage'].reindex(pivot.columns).values

# 1.0 = perfectly healthy cadence,
# 0.0 = fully offline. Values between = partial outages.
adjusted_coverage = np.clip(actual_coverage / expected_coverage, 0.0, 1.0)

downtime_ratio = pd.Series(
    1.0 - adjusted_coverage,
    index=pivot.columns,
    name='downtime_ratio',
)


# Outage duration: only gaps LARGER than the cadence count as outages.
# A gap of 17 min between readings from a 17-min station is normal.
# A gap of 6 hours is an outage.# -------------------------------------------------------------
def outage_stats(series, expected_gap_min):
    """Return (max_outage_hours, total_outage_hours) for one station."""
    valid = series.dropna()
    if len(valid) < 2:
        return 0.0, 0.0

    gaps_min = valid.index.to_series().diff().dt.total_seconds().dropna() / 60
    threshold = 3.0 * expected_gap_min    # 3× cadence = start of outage
    long_gaps = gaps_min[gaps_min > threshold]

    if len(long_gaps) == 0:
        return 0.0, 0.0

    # Subtract the expected cadence from each gap to isolate "extra" outage time
    extra_min = (long_gaps - expected_gap_min)
    return float(extra_min.max() / 60.0), float(extra_min.sum() / 60.0)


max_outage, total_outage = {}, {}
for st in pivot.columns:
    gap_min = cadence.loc[st, 'median_gap_min']
    mx, tot = outage_stats(pivot[st], gap_min)
    max_outage[st]  = mx
    total_outage[st] = tot

downtime = pd.DataFrame({
    'downtime_ratio':                downtime_ratio,
    'max_continuous_downtime_hours': pd.Series(max_outage,   name='max_continuous_downtime_hours'),
    'total_downtime_hours':          pd.Series(total_outage, name='total_downtime_hours'),
})


print(f"    Downtime ratio      : mean={downtime['downtime_ratio'].mean():.3f}  "
      f"median={downtime['downtime_ratio'].median():.3f}  "
      f"max={downtime['downtime_ratio'].max():.3f}")
print(f"    Max outage (hours)  : mean={downtime['max_continuous_downtime_hours'].mean():.1f}  "
      f"max={downtime['max_continuous_downtime_hours'].max():.1f}")
print(f"    Total outage (hours): mean={downtime['total_downtime_hours'].mean():.1f}  "
      f"max={downtime['total_downtime_hours'].max():.1f}")

# Show the top 5 worst stations by downtime ratio as a preview
print("\n    Top 5 by downtime ratio:")
print(downtime['downtime_ratio'].sort_values(ascending=False).head(5).round(3).to_string())


Downtime metrics (cadence-adjusted, per-station active period) ...
    Downtime ratio      : mean=0.268  median=0.286  max=0.583
    Max outage (hours)  : mean=54.9  max=735.5
    Total outage (hours): mean=1478.0  max=2370.9

    Top 5 by downtime ratio:
Boni Serrano                0.583
Near Puregold Tayuman       0.485
Sta. Ana Hospital           0.485
Anda Circle                 0.485
San Sebastian Residences    0.483


In [66]:
print("Mean imputation ...")
pivot_imp = pivot.fillna(pivot.mean())

Mean imputation ...


#### **FFT Spectral Features**

In [67]:
print("\nFFT spectral features ...")

fft_rows = []
for st in pivot_imp.columns:
    x = pivot_imp[st].values.astype(float)
    x = x - x.mean()
    mag = np.abs(np.fft.rfft(x))
    power = mag ** 2
    if power.sum() > 0:
        p = power / power.sum()
        ent = -np.sum(p * np.log(p + 1e-12))
    else:
        ent = 0.0
    fft_rows.append({
        'station':              st,
        'fft_energy':           float(power.sum()),
        'fft_max_amp':          float(mag[1:].max()) if len(mag) > 1 else 0.0,
        'fft_spectral_entropy': float(ent),
    })

fft_df = pd.DataFrame(fft_rows).set_index('station')


FFT spectral features ...


#### **Station Feature Matrix**

In [68]:
station_features = downtime.join(fft_df).fillna(0.0)

if "density_persons/sqkm" in df.columns:
    density = df.groupby("station").agg({"density_persons/sqkm": "first"})
    station_features = station_features.join(density).fillna(0.0)

print(station_features.shape)
print(station_features.head())

(67, 7)
                        downtime_ratio  max_continuous_downtime_hours  \
3S Center Bagbaguin           0.287746                      22.050333   
3S Center Paso De Blas        0.286557                      22.050333   
3S Center Ugong               0.296493                      22.040500   
ASMPH                         0.189589                       5.904667   
Along Shaw Blvd.              0.331757                      12.875333   

                        total_downtime_hours    fft_energy   fft_max_amp  \
3S Center Bagbaguin              2007.627000  1.105508e+11  30055.799622   
3S Center Paso De Blas           2001.042000  1.149209e+11  27826.316921   
3S Center Ugong                  2152.245000  1.270705e+11  26933.977171   
ASMPH                             865.548667  4.777552e+09   6040.076719   
Along Shaw Blvd.                 1260.595333  1.712141e+10  10089.058441   

                        fft_spectral_entropy  density_persons/sqkm  
3S Center Bagbaguin        

# **Modeling**

### **Model 1: Ensemble (Isolation Forest + OC-SVM)**

In [69]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(station_features)

def minmax01(x):
    x = np.asarray(x, dtype=float)
    if x.max() - x.min() < 1e-12: return np.zeros_like(x)
    return (x - x.min()) / (x.max() - x.min())

iso_forest = IsolationForest(contamination=0.1, random_state=RANDOM_SEED)
iso_raw    = -iso_forest.fit_predict(X_scaled)   # 1 = outlier, 0 = inlier

oc_svm     = OneClassSVM(kernel="rbf", gamma="scale", nu=0.1)
svm_raw    = -oc_svm.fit_predict(X_scaled)

ensemble_raw = 0.5 * minmax01(iso_raw) + 0.5 * minmax01(svm_raw)

In [70]:
#apply downtime penalty
penalty = (1.0
           + 3.0 * station_features["downtime_ratio"]
           + station_features["max_continuous_downtime_hours"] / 50.0)

station_features["model1_risk_score"] = ensemble_raw * penalty

model1_ranking = (station_features[["model1_risk_score", "downtime_ratio",
                                     "max_continuous_downtime_hours"]]
                  .sort_values("model1_risk_score", ascending=False))
print("=== MODEL 1: OC-SVM + IF ENSEMBLE — TOP 10 ===")
print(model1_ranking.head(10).round(4).to_string())
model1_ranking.to_csv(f"{OUT_M1}/ranking.csv")

=== MODEL 1: OC-SVM + IF ENSEMBLE — TOP 10 ===
                                              model1_risk_score  downtime_ratio  max_continuous_downtime_hours
Boni Serrano                                             8.7296          0.5828                       735.5387
Brgy. Benedicto, Jaro, Iloilo                            7.4905          0.0604                       690.0000
Demo Unit 4                                              4.4739          0.4610                       328.2380
Brgy. Ambassador                                         2.9411          0.0874                       231.0000
Tagaytay (Neogan)                                        2.7664          0.0910                       213.0000
Sta. Ana Hospital                                        1.5100          0.4853                        28.2087
MNHS AQ Station                                          1.2200          0.3330                        22.0445
Quezon Rd, San Simon                                     1.2016  

#### **Model 2: ECOD (Empirical-Cumulative-distribution-based Outlier Detection)**

In [71]:
ecod = ECOD()
ecod.fit(X_scaled)
ecod_raw = minmax01(ecod.decision_scores_)

In [72]:
# apply downtime penalty
station_features["model2_risk_score"] = ecod_raw * penalty

model2_ranking = (station_features[["model2_risk_score", "downtime_ratio",
                                     "max_continuous_downtime_hours"]]
                  .sort_values("model2_risk_score", ascending=False))
print("=== MODEL 2: ECOD — TOP 10 ===")
print(model2_ranking.head(10).round(4).to_string())
model2_ranking.to_csv(f"{OUT_M2}/ranking.csv")

=== MODEL 2: ECOD — TOP 10 ===
                               model2_risk_score  downtime_ratio  max_continuous_downtime_hours
Boni Serrano                             14.1457          0.5828                       735.5387
Brgy. Benedicto, Jaro, Iloilo            12.2260          0.0604                       690.0000
Brgy. Ambassador                          4.6802          0.0874                       231.0000
Demo Unit 4                               4.4372          0.4610                       328.2380
Tagaytay (Neogan)                         2.4960          0.0910                       213.0000
MNHS AQ Station                           1.8510          0.3330                        22.0445
Quezon Rd, San Simon                      1.7148          0.1919                        41.3722
Ayala Ave.                                1.2935          0.4716                        27.8737
Illumina                                  1.2898          0.0332                        91.0000
Manila Ob

#### **Model 3: Autoencoder with Regression**

**AER Model Configurations**

In [73]:
RESAMPLE = "20min"

N_STEPS          = 48
B_UNITS          = 30
GAMMA            = 0.5
MASK_RATIO       = 0.01
EWMA_SPAN_RATIO  = 0.1
THRESH_SIG       = 4.0
COMBINE          = 'MULT'

# Training
EPOCHS      = 5
BATCH       = 256
TRAIN_FRAC  = 0.8
VAL_SPLIT   = 0.1

# Extensions
USE_FFT_CHANNELS = True
FFT_KEEP_RATIO   = 0.10

# Station gating (from coverage evidence)
MAX_TRAIN_WINDOWS = 8000     
MIN_N_RECORDS      = 500
MIN_COVERAGE_20MIN = 0.30
MIN_TRAIN_WINDOWS  = 100

# Risk aggregation weights
RISK_WEIGHTS = {
    'downtime_ratio':                 0.25,
    'max_continuous_downtime_hours':  0.15,
    'anomaly_freq':                   0.25,
    'anomaly_days_ratio':             0.15,
    'mean_anomaly_score':             0.10,
    'fft_energy':                     0.05,
    'fft_spectral_entropy':           0.05,
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("=" * 76)
print("AER PIPELINE — PM2.5 Sensor-Maintenance Prioritization")
print("=" * 76)
print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Resolution : {RESAMPLE}  (n={N_STEPS} → {N_STEPS * 20 / 60:.1f} h window)")
print(f"AER        : b={B_UNITS}, γ={GAMMA}, combine={COMBINE}")
print("=" * 76)

AER PIPELINE — PM2.5 Sensor-Maintenance Prioritization
TensorFlow : 2.22.0-rc0
GPU        : False
Resolution : 20min  (n=48 → 16.0 h window)
AER        : b=30, γ=0.5, combine=MULT


**Pre-processing Helpers**

In [74]:
def detrend(x):
    """Least-squares linear detrend — paper Sec III-A."""
    t = np.arange(len(x), dtype=float)
    A = np.vstack([t, np.ones_like(t)]).T
    coef, *_ = np.linalg.lstsq(A, x, rcond=None)
    return x - (A @ coef)

def scale_neg1_1(x):
    """Min-max to [-1, 1] — paper Sec III-A."""
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return 2.0 * (x - mn) / (mx - mn) - 1.0

def build_fft_channels(x_detrended, keep_ratio=FFT_KEEP_RATIO):
    """Split detrended signal into [raw, seasonal, residual] via FFT low-pass."""
    n = len(x_detrended)
    fft = np.fft.rfft(x_detrended)
    k = max(1, int(len(fft) * keep_ratio))
    filt = fft.copy(); filt[k:] = 0.0
    seasonal = np.fft.irfft(filt, n=n)
    residual = x_detrended - seasonal
    return np.column_stack([x_detrended, seasonal, residual])

def make_windows(arr, n_steps, train_frac, max_windows=MAX_TRAIN_WINDOWS):
    T = arr.shape[0]
    split = int(T * train_frac)

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(arr[:split])
    scaled = scaler.transform(arr).astype(np.float32)
    target = scaled[:, 0]

    X, Y = [], []
    for i in range(1, split - n_steps - 1):
        X.append(scaled[i:i + n_steps])
        y = np.zeros((n_steps + 2, 1), dtype=np.float32)
        y[0, 0]               = target[i - 1]
        y[1:n_steps + 1, 0]   = target[i:i + n_steps]
        y[n_steps + 1, 0]     = target[i + n_steps]
        Y.append(y)

    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)

    if len(X) > max_windows:
        idx = np.random.choice(len(X), max_windows, replace=False)
        X = X[idx]
        Y = Y[idx]

    return X, Y, scaled, scaler

def build_aer(n_steps, n_features, b=30, gamma=0.5):
    """AER model — paper Sec V-A, Eq. 5."""
    inp = Input(shape=(n_steps, n_features), name='aer_input')
    enc = layers.Bidirectional(layers.LSTM(b), name='encoder')(inp)
    dec = layers.RepeatVector(n_steps + 2, name='repeat_plus2')(enc)
    dec = layers.Bidirectional(
        layers.LSTM(b, return_sequences=True), name='decoder')(dec)
    out = layers.TimeDistributed(layers.Dense(1), name='output')(dec)
    model = Model(inp, out)

    def aer_loss(y_true, y_pred):
        rev_t, rec_t, fwd_t = (y_true[:, 0, 0],
                               y_true[:, 1:n_steps + 1, 0],
                               y_true[:, n_steps + 1, 0])
        rev_p, rec_p, fwd_p = (y_pred[:, 0, 0],
                               y_pred[:, 1:n_steps + 1, 0],
                               y_pred[:, n_steps + 1, 0])
        v_pred = 0.5 * tf.reduce_mean(tf.square(rev_t - rev_p)) \
               + 0.5 * tf.reduce_mean(tf.square(fwd_t - fwd_p))
        v_rec  = tf.reduce_mean(tf.square(rec_t - rec_p))
        return gamma * v_pred + (1.0 - gamma) * v_rec

    model.compile(optimizer='adam', loss=aer_loss)
    return model

def minmax01(x):
    x = np.nan_to_num(np.asarray(x, dtype=float), nan=0.0)
    mn, mx = x.min(), x.max()
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def dynamic_threshold(scores, n_sigma=4.0):
    """Non-parametric dynamic threshold — paper Sec III-D."""
    s = np.asarray(scores, dtype=float)
    T = len(s)
    if T < 5:
        return np.zeros(T, dtype=bool)
    win = max(10, T // 3)
    step = max(1, win // 10)
    flags = np.zeros(T, dtype=bool)
    for start in range(0, T, step):
        end = min(start + win, T)
        w = s[start:end]
        if len(w) < 3:
            continue
        thr = w.mean() + n_sigma * w.std()
        flags[start:end] |= s[start:end] > thr
    return flags

def score_station(arr_scaled, model, n_steps, timestamps):
    """Bi-directional + reconstruction + combination — paper Sec V-B/C/D."""
    T = arr_scaled.shape[0]
    target = arr_scaled[:, 0]

    windows = np.stack([arr_scaled[i:i + n_steps]
                        for i in range(T - n_steps)], axis=0).astype(np.float32)
    preds = model.predict(windows, batch_size=512, verbose=0)

    alpha_fwd = np.full(T, np.nan)
    alpha_rev = np.full(T, np.nan)
    rec_buf   = [[] for _ in range(T)]

    for i in range(T - n_steps):
        rev_p = preds[i, 0, 0]
        rec_p = preds[i, 1:n_steps + 1, 0]
        fwd_p = preds[i, n_steps + 1, 0]

        if i + n_steps < T:
            alpha_fwd[i + n_steps] = abs(target[i + n_steps] - fwd_p)
        if i - 1 >= 0:
            alpha_rev[i - 1] = abs(target[i - 1] - rev_p)

        for j in range(n_steps):
            if i + j < T:
                rec_buf[i + j].append(rec_p[j])

    rec_med = np.array([np.median(v) if v else np.nan for v in rec_buf])
    alpha_rec = np.nan_to_num(np.abs(target - rec_med), nan=0.0)

    # Masking (Sec V-B)
    m = max(1, int(MASK_RATIO * T))
    alpha_fwd = np.nan_to_num(alpha_fwd, nan=0.0)
    alpha_rev = np.nan_to_num(alpha_rev, nan=0.0)
    alpha_fwd[:m] = 0.0
    if len(alpha_rev):
        alpha_rev[:m] = alpha_rev.min()

    # Bi-directional (Eq. 6)
    alpha_b = np.zeros(T)
    for i in range(T):
        if   i < n_steps + m:   alpha_b[i] = alpha_rev[i]
        elif i < T - n_steps:   alpha_b[i] = 0.5 * alpha_rev[i] + 0.5 * alpha_fwd[i]
        else:                   alpha_b[i] = alpha_fwd[i]

    # Combination (Eq. 7–10)
    b01, r01 = minmax01(alpha_b), minmax01(alpha_rec)
    if   COMBINE == 'MULT': alpha_c = (r01 + 1.0) * (b01 + 1.0)
    elif COMBINE == 'SUM':  alpha_c = 0.5 * b01 + 0.5 * r01
    elif COMBINE == 'PRED': alpha_c = b01
    elif COMBINE == 'REC':  alpha_c = r01
    else: raise ValueError(COMBINE)

    # EWMA (Sec III-C)
    span = max(2, int(EWMA_SPAN_RATIO * T))
    alpha_s = pd.Series(alpha_c).ewm(span=span, adjust=False).mean().values

    # Threshold (Sec III-D)
    flags = dynamic_threshold(alpha_s, THRESH_SIG)

    daily_flags = pd.Series(flags, index=timestamps).resample('1D').max()

    return {
        'scores':      alpha_s,
        'flags':       flags,
        'alpha_b':     alpha_b,
        'alpha_rec':   alpha_rec,
        'daily_flags': daily_flags,
    }

**Train + Score Per Station**

In [75]:
print(f"\nTraining AER per station "
      f"(n={N_STEPS}, b={B_UNITS}, γ={GAMMA}, epochs={EPOCHS}) ...")

results = []
t_start = time.time()

for k, station in enumerate(pivot_imp.columns):
    t0 = time.time()
    x_raw = pivot_imp[station].values.astype(float)
    # print(f"Processing station {k+1}: {station} | Timeline length: {len(x_raw)} rows") # for monitoring

    x_detrended = detrend(x_raw)
    channels_raw = (build_fft_channels(x_detrended, FFT_KEEP_RATIO)
                    if USE_FFT_CHANNELS else x_detrended[:, None])

    channels = np.column_stack([
        scale_neg1_1(channels_raw[:, c])
        for c in range(channels_raw.shape[1])
    ])

    X, Y, arr_scaled, scaler = make_windows(channels, N_STEPS, TRAIN_FRAC)
    if len(X) < MIN_TRAIN_WINDOWS:
        print(f"  [{k+1:>3}/{len(pivot_imp.columns)}] "
              f"{station[:40]:<42} SKIP ({len(X)} windows)")
        continue

    model = build_aer(N_STEPS, channels.shape[1], B_UNITS, GAMMA)

    early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=1,          
    restore_best_weights=True
    )
    
    hist = model.fit(X, Y, epochs=EPOCHS, batch_size=BATCH,
                     validation_split=VAL_SPLIT, verbose=0, shuffle=True, callbacks=[early_stopping])

    sc = score_station(arr_scaled, model, N_STEPS, pivot.index)

    results.append({
        'station':            station,
        'cluster':            cadence.loc[station, 'cluster']
                              if station in cadence.index else 'unknown',
        'n_bins':             int(len(x_raw)),
        'anomaly_count':      int(sc['flags'].sum()),
        'anomaly_freq':       float(sc['flags'].mean()),
        'anomaly_days_ratio': float(sc['daily_flags'].mean())
                              if len(sc['daily_flags']) else 0.0,
        'mean_anomaly_score': float(sc['scores'][sc['flags']].mean())
                              if sc['flags'].any() else 0.0,
        'max_anomaly_score':  float(sc['scores'].max()),
        'train_loss':         float(hist.history['loss'][-1]),
        'val_loss':           float(hist.history['val_loss'][-1]),
        'n_train_windows':    int(len(X)),
    })

    safe = station.replace('/', '_').replace(',', '_')[:60]
    pd.DataFrame({
        'timestamp':            pivot.index,
        'score':                sc['scores'],
        'is_anomaly':           sc['flags'].astype(int),
        'alpha_bidirectional':  sc['alpha_b'],
        'alpha_reconstruction': sc['alpha_rec'],
    }).to_csv(f'{OUT_M3}/aer_scores_{safe}.csv', index=False)

    del model, X, Y, arr_scaled, sc, channels, channels_raw
    tf.keras.backend.clear_session()

    elapsed = time.time() - t0
    total_min = (time.time() - t_start) / 60
    print(f"  [{k+1:>3}/{len(pivot_imp.columns)}] "
          f"{station[:40]:<42} "
          f"anom_freq={results[-1]['anomaly_freq']:.3f} "
          f"({elapsed:.1f}s / {total_min:.1f}min)")

print(f"\nTrained {len(results)} stations in "
      f"{(time.time() - t_start)/60:.1f} min")


Training AER per station (n=48, b=30, γ=0.5, epochs=5) ...
  [  1/67] 3S Center Bagbaguin                        anom_freq=0.013 (33.2s / 0.6min)
  [  2/67] 3S Center Paso De Blas                     anom_freq=0.009 (38.0s / 1.2min)
  [  3/67] 3S Center Ugong                            anom_freq=0.009 (32.1s / 1.7min)
  [  4/67] ASMPH                                      anom_freq=0.009 (34.2s / 2.3min)
  [  5/67] Along Shaw Blvd.                           anom_freq=0.008 (31.8s / 2.8min)
  [  6/67] Anda Circle                                anom_freq=0.006 (31.5s / 3.3min)
  [  7/67] Aurora Boulevard (Tramo) - Andrews Avenu   anom_freq=0.013 (33.1s / 3.9min)
  [  8/67] Ayala Ave.                                 anom_freq=0.008 (22.3s / 4.3min)
  [  9/67] Barangay Buli - Concepcion Street          anom_freq=0.010 (18.2s / 4.6min)
  [ 10/67] Barangay Fortune - Santan Street           anom_freq=0.010 (37.1s / 5.2min)
  [ 11/67] Barangay San Antonio - Parañaque-Sucat R   anom_freq=0.010 

W0000 00:00:1790434085.870983  550973 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1790434065022868000 in buffer 'PrefetchDataset' sat unconsumed for 20.8473 seconds!


  [ 12/67] Barangay San Rafael Hall                   anom_freq=0.010 (45.6s / 21.9min)
  [ 13/67] Barangay Tambo - Quirino Avenue            anom_freq=0.012 (29.4s / 22.4min)
  [ 14/67] Barangay Tanza 2 Hall                      anom_freq=0.010 (29.8s / 22.9min)
  [ 15/67] Barangay Tunasan - Maharlika Highway (SM   anom_freq=0.010 (27.9s / 23.4min)
  [ 16/67] Bataan Tourism Park                        anom_freq=0.009 (16.2s / 23.6min)
  [ 17/67] Bo. Luz Barangay Hall, Limay               anom_freq=0.008 (13.7s / 23.9min)
  [ 18/67] Boni Serrano                               anom_freq=0.008 (13.5s / 24.1min)
  [ 19/67] Bradco Ave. cor Macapagal                  anom_freq=0.015 (13.4s / 24.3min)
  [ 20/67] Brgy. Ambassador                           anom_freq=0.009 (13.4s / 24.5min)
  [ 21/67] Brgy. Benedicto, Jaro, Iloilo              anom_freq=0.000 (16.8s / 24.8min)
  [ 22/67] C3 Road (5th Avenue) - Rizal Avenue Exte   anom_freq=0.008 (19.3s / 25.1min)
  [ 23/67] Caloocan Sports Compl

W0000 00:00:1790434393.630278  550973 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1790434382706302000 in buffer 'PrefetchDataset' sat unconsumed for 10.924 seconds!


  [ 28/67] Illumina                                   anom_freq=0.010 (45.8s / 27.2min)


...

  [ 65/67] Valenzuela CENRO (ARCA North)              anom_freq=0.010 (56.7s / 47.1min)


W0000 00:00:1790435636.865685  550973 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1790435621807490000 in buffer 'PrefetchDataset' sat unconsumed for 15.0582 seconds!


  [ 66/67] Valle1                                     anom_freq=0.009 (58.6s / 48.1min)


W0000 00:00:1790435691.165927  550973 prefetch_dataset_op.cc:476] SEVERE STARVATION: Element UID 1790435680182946000 in buffer 'PrefetchDataset' sat unconsumed for 10.983 seconds!


  [ 67/67] World Trade Center                         anom_freq=0.012 (52.7s / 48.9min)

Trained 67 stations in 48.9 min


In [76]:
# RISK AGGREGATION FOR MODEL 3
res_df = pd.DataFrame(results).set_index("station")
risk3  = res_df.join(downtime).join(fft_df).fillna(0.0)

feat_cols = list(RISK_WEIGHTS.keys())
risk_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(risk3[feat_cols].fillna(0.0)),
    columns=[c + "_norm" for c in feat_cols],
    index=risk3.index,
)
risk3["model3_risk_score"] = sum(
    risk_norm[f"{c}_norm"] * w for c, w in RISK_WEIGHTS.items()
)
risk3 = risk3.sort_values("model3_risk_score", ascending=False)
risk3.to_csv(f"{OUT_M3}/ranking.csv")

print("=== MODEL 3: AER — TOP 10 ===")
show = ["model3_risk_score", "downtime_ratio", "anomaly_freq",
        "anomaly_days_ratio", "mean_anomaly_score", "fft_spectral_entropy"]
print(risk3[show].head(10).round(4).to_string())

=== MODEL 3: AER — TOP 10 ===
                                           model3_risk_score  downtime_ratio  anomaly_freq  anomaly_days_ratio  mean_anomaly_score  fft_spectral_entropy
station                                                                                                                                                 
Boni Serrano                                          0.6990          0.5828        0.0077              0.0100              1.0442                9.0029
Caloocan Sports Complex                               0.6460          0.2862        0.0176              0.0219              1.0626                8.8675
Near Katipunan Ave., Quezon City                      0.6157          0.2844        0.0168              0.0210              1.0517                8.4137
Bradco Ave. cor Macapagal                             0.5740          0.2927        0.0148              0.0183              1.0774                8.4285
3S Center Bagbaguin                                 

**Model Comparison**

In [77]:
# Model Comparison
comparison = pd.DataFrame(index=station_features.index)

# Model 1
comparison["model1_score"] = station_features["model1_risk_score"]
comparison["model1_rank"]  = comparison["model1_score"].rank(ascending=False)

# Model 2
comparison["model2_score"] = station_features["model2_risk_score"]
comparison["model2_rank"]  = comparison["model2_score"].rank(ascending=False)

# Model 3 
comparison["model3_score"] = risk3["model3_risk_score"]
comparison["model3_rank"]  = comparison["model3_score"].rank(ascending=False)
comparison["cluster"]      = cadence["cluster"]
comparison["downtime_ratio"] = downtime["downtime_ratio"]

comparison = comparison.sort_values("model3_rank")
comparison.to_csv(f"{OUT_PROOF}/model_comparison.csv")

print("=== SIDE-BY-SIDE RANKING (top 15 by Model 3) ===")
print(comparison[["model1_rank", "model2_rank", "model3_rank",
                  "cluster", "downtime_ratio"]].head(15).round(2).to_string())

=== SIDE-BY-SIDE RANKING (top 15 by Model 3) ===
                                           model1_rank  model2_rank  model3_rank      cluster  downtime_ratio
Boni Serrano                                       1.0          1.0          1.0  A_high_freq            0.58
Caloocan Sports Complex                           40.0         43.0          2.0  B_candidate            0.29
Near Katipunan Ave., Quezon City                  40.0         24.0          3.0  A_high_freq            0.28
Bradco Ave. cor Macapagal                         40.0         39.0          4.0  A_high_freq            0.29
3S Center Bagbaguin                               40.0         50.0          5.0  B_candidate            0.29
Aurora Boulevard (Tramo) - Andrews Avenue         40.0         55.0          6.0  B_candidate            0.29
Malabon Action Center                             40.0         15.0          7.0  B_candidate            0.29
Barangay Tambo - Quirino Avenue                   40.0         58.0    

In [78]:
# SPEARMAN RANK CORRELATION
print("\nSpearman rank correlation between models:")
print(f"  Model1 vs Model2: {comparison[['model1_rank','model2_rank']].corr(method='spearman').iloc[0,1]:.3f}")
print(f"  Model1 vs Model3: {comparison[['model1_rank','model3_rank']].corr(method='spearman').iloc[0,1]:.3f}")
print(f"  Model2 vs Model3: {comparison[['model2_rank','model3_rank']].corr(method='spearman').iloc[0,1]:.3f}")


Spearman rank correlation between models:
  Model1 vs Model2: 0.556
  Model1 vs Model3: -0.300
  Model2 vs Model3: -0.291


#### **Sensor Tiers**

In [79]:
# TIER THRESHOLDS  (from EMB MC 2021-14)
d_critical   = 0.25   # below 75% capture
d_degraded   = 0.10   # below 90% capture
d_occasional = 0.05   # below 95% capture

# Use Model 3's anomaly_freq (temporal anomalies) as the "suspicious" trigger
risk_final = risk3.copy()
a_suspicious = risk_final["anomaly_freq"].quantile(0.90)

def health_tier(row):
    d = row["downtime_ratio"]
    a = row["anomaly_freq"]
    if d > d_critical:   return "CRITICAL_OFFLINE"
    if d > d_degraded:   return "DEGRADED_OFFLINE"
    if a > a_suspicious: return "SUSPICIOUS_READINGS"
    if d > d_occasional: return "OCCASIONAL_OUTAGES"
    return "HEALTHY"

risk_final["health_tier"] = risk_final.apply(health_tier, axis=1)
risk_final.to_csv(f"{OUT_PROOF}/final_action_list.csv")

print("=== HEALTH TIER DISTRIBUTION ===")
print(risk_final["health_tier"].value_counts().to_string())

print("\n=== ACTION LIST BY TIER ===")
cols = ["model3_risk_score", "downtime_ratio", "anomaly_freq",
        "max_continuous_downtime_hours"]
for tier in ["CRITICAL_OFFLINE", "DEGRADED_OFFLINE",
             "SUSPICIOUS_READINGS", "OCCASIONAL_OUTAGES"]:
    sub = risk_final[risk_final["health_tier"] == tier]
    if len(sub) == 0: continue
    print(f"\n{tier} ({len(sub)} stations)")
    print(sub[cols].head(10).round(3).to_string())

=== HEALTH TIER DISTRIBUTION ===
health_tier
CRITICAL_OFFLINE      50
HEALTHY                8
DEGRADED_OFFLINE       6
OCCASIONAL_OUTAGES     3

=== ACTION LIST BY TIER ===

CRITICAL_OFFLINE (50 stations)
                                           model3_risk_score  downtime_ratio  anomaly_freq  max_continuous_downtime_hours
station                                                                                                                  
Boni Serrano                                           0.699           0.583         0.008                        735.539
Caloocan Sports Complex                                0.646           0.286         0.018                         22.050
Near Katipunan Ave., Quezon City                       0.616           0.284         0.017                         21.874
Bradco Ave. cor Macapagal                              0.574           0.293         0.015                         21.872
3S Center Bagbaguin                                    0.545  

#### **Insights and Recommendations**

**Model 1 and 2** show moderate positive agreement ($\rho=0.556$) because both evaluate static, station-level summary features (risk score, downtime ratios, and maximum outage duration). They identified macro-level hardware failures and prolonged network outages, consistently ranking severely offline stations like Boni Serrano and Brgy. Benedicto, Jaro, Iloilo as top stations that requires frequent monitoring.

**Model 3** evaluates sequential time-series windows using the Bidirectional LSTM with FFT decomposition, EWMA Smoothing, and dynamic thresholding. The AER model produces unique ranks across the 67 stations and captured station level features (downtime_ratio, anomaly_frequency, anomaly_days_ratio, mean_anomaly_score, ftf_spectral_entropy.)

**Based on the health tier created:**
* **CRITICAL_OFFLINE:**       50 stations
* **HEALTHY:**                 7 stations
* **DEGRADED_OFFLINE:**        6 stations
* **SUSPICIOUS_READINGS:**     2 stations
* **OCCASIONAL_OUTAGE:**      2 stations

